In [ ]:
# [Setup]

# In VSCodium :
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)

In [ ]:
# 02_A_Source_Identification_And_Sorting_By_Algorithm.ipynb : Cell 1

from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

from photutils.detection import DAOStarFinder, find_peaks
from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats, aperture_photometry

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Paths
cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts"
)

manifest_path = cutout_dir.parent / "plate_manifest.csv"

# Detection parameters
FWHM = 3.0
THRESHOLD_SIGMA = 5.0
MIN_SOURCES_DAO = 5 # fallback threshold
BOX_SIZE = 11 # for find_peaks fallback

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts to process")

# Plate limiting magnitude lookup (lim_mag_apass / lim_mag_atlas), sourced
# from 01_Load_Cutout.ipynb's plate_manifest.csv. Same fast-path pattern
# that notebook already uses for observation dates : one CSV read instead
# of touching every FITS header, and no network calls (this data was
# already fetched by 01's Cell 1, from sess.exposures()). Keyed by
# filename so Cell 3 can look it up per-plate with zero extra I/O.
plate_limit_lookup = {}
if manifest_path.exists():
    manifest_df_for_limits = pd.read_csv(manifest_path)
    has_cols = (
        "filename" in manifest_df_for_limits.columns
        and ("lim_mag_apass" in manifest_df_for_limits.columns
             or "lim_mag_atlas" in manifest_df_for_limits.columns)
    )
    if has_cols:
        for _, row in manifest_df_for_limits.iterrows():
            fname = row.get("filename")
            if not (isinstance(fname, str) and fname):
                continue
            apass_val = row.get("lim_mag_apass")
            atlas_val = row.get("lim_mag_atlas")
            plate_limit_lookup[Path(fname).name] = {
                "lim_mag_apass": float(apass_val) if pd.notna(apass_val) else None,
                "lim_mag_atlas": float(atlas_val) if pd.notna(atlas_val) else None,
            }
        n_with_limits = sum(
            1 for v in plate_limit_lookup.values()
            if v["lim_mag_apass"] is not None or v["lim_mag_atlas"] is not None
        )
        print(f"Loaded plate limiting magnitudes for {n_with_limits} of {len(plate_limit_lookup)} manifest rows.")
    else:
        print("Manifest found but missing 'filename'/'lim_mag_*' columns -- plate limits unavailable.")
else:
    print("No plate_manifest.csv found next to cutouts -- plate limits unavailable.")

def get_plate_limits(fits_path):
    """Returns (lim_mag_apass, lim_mag_atlas) for a cutout, or (None, None)
    if that filename isn't in the manifest lookup."""
    entry = plate_limit_lookup.get(fits_path.name)
    if entry is None:
        return None, None
    return entry["lim_mag_apass"], entry["lim_mag_atlas"]

Found 5398 cutouts to process
Loaded plate limiting magnitudes for 2795 of 5398 manifest rows.


In [ ]:
# 02_A_Source_Identification_And_Sorting_By_Algorithm.ipynb : Cell 2

from photutils.aperture import CircularAperture, aperture_photometry

def detect_sources(fits_path, aperture_radius=5.0):
    """
    Detect stars and compute fixed-aperture photometry for APASS calibration.
    """
    
    # Load FITS
    data = fits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))

    # Background estimation
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    data_sub = data - median # Subtracting the median from the plate

    # Source detection (DAOStarFinder first)
    try:
        daofind = DAOStarFinder(
            fwhm=FWHM,
            threshold=THRESHOLD_SIGMA * std,
            sharpness_range=(0.2, 2.0)
        )
        sources = daofind(data_sub)
        algorithm = "DAOStarFinder"

    except Exception:
        sources = None
        algorithm = "DAOStarFinder_failed"

    # Fallback to find_peaks if needed
    if sources is None or len(sources) < MIN_SOURCES_DAO:
        sources = find_peaks(
            data_sub,
            threshold=THRESHOLD_SIGMA * std,
            box_size=BOX_SIZE
        )
        algorithm = "find_peaks"

    if sources is None or len(sources) == 0:
        return None, algorithm, data, data_sub, std, None, None

    # Choose centroid columns
    if "x_centroid" in sources.colnames:
        x_col, y_col = "x_centroid", "y_centroid"
    else:
        x_col, y_col = "x_peak", "y_peak"

    positions = np.transpose((sources[x_col], sources[y_col]))

    # Fixed aperture photometry
    apertures = CircularAperture(positions, r=aperture_radius)
    phot_table = aperture_photometry(data_sub, apertures)
    
    sources["aperture_flux"] = phot_table["aperture_sum"]
    return sources, algorithm, data, data_sub, std, x_col, y_col

In [ ]:
# 02_A_Source_Identification_And_Sorting_By_Algorithm.ipynb : Cell 3

%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output

import pickle
import time
from io import BytesIO
from pathlib import Path
from threading import Thread
from concurrent.futures import ThreadPoolExecutor, as_completed

from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from photutils.centroids import centroid_sources, centroid_com
from photutils.profiles import RadialProfile

from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, match_coordinates_sky, search_around_sky
import astropy.units as u

from astroquery.gaia import Gaia
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import ndimage
from skimage.transform import hough_line, hough_line_peaks
import warnings

APERTURE_RADIUS = 5.0

# Target — V Coronae Australis (V CrA), an R Coronae Borealis-type variable
# in Corona Australis. Coordinates from Wikipedia's infobox (J2000/ICRS,
# sourced from Gaia DR3): RA 18h47m32.30962s, Dec -38d09m32.3079s.
TARGET_STAR_NAME = "V CrA"
TARGET_STAR = SkyCoord(
    ra=281.884623417 * u.deg,
    dec=-38.158974417 * u.deg
)

# Bump this whenever target-detection logic changes. On the next prescan(),
# any cached plate whose stored target_version doesn't match gets a cheap
# re-check (reopen FITS, no network calls) instead of a full rescan.
TARGET_DETECTION_VERSION = 2

# Bump this whenever quality-classification logic changes. On the next
# prescan(), any cached plate whose stored quality_version doesn't match
# gets its quality re-derived offline (reopen FITS, recompute defects,
# reclassify).
QUALITY_VERSION = 2

# Bump this whenever the per-source photometry pipeline changes (centroid
# refinement, adaptive apertures, local background, saturation detection,
# etc.). On the next full_scan(), any cached plate whose stored
# photometry_version doesn't match gets its photometry fully redone, even
# if it was already at phase == "full" under the old pipeline.
PHOTOMETRY_VERSION = 1

# Number of threads prescan() uses to process plates in parallel. The
# per-plate work (Hough transform, ndimage labeling, FITS I/O) is all
# numpy/scipy/skimage compiled code that releases Python's GIL while it
# runs, so a thread pool gives close to real multi-core speedup here
# without the Windows/Jupyter multiprocessing pickling headaches. Lower
# this if your machine has fewer cores or you hit memory pressure (each
# worker holds one full image array in memory at a time).
PRESCAN_WORKERS = 8

# Cache paths
# Cache directory derived from cutout_dir (set in Cell 1) rather than
# hardcoded, so each target/survey (R CrB, V CrA, ...) automatically gets
# its own isolated plate_db/paradigm_db/Gaia-APASS-SIMBAD caches instead
# of accumulating everything into one shared, ever-growing cache that
# would mix plates from unrelated surveys into the same Filter/dropdown
# lists (the paradigm picker and main viewer both iterate over ALL of
# plate_db, not just the current cutouts list).
gaia_dir = cutout_dir.parent / "gaia"
gaia_dir.mkdir(parents=True, exist_ok=True)

PLATE_DB_FILE = gaia_dir / "plate_db.pkl"
GAIA_CACHE_FILE = gaia_dir / "gaia_cache.pkl"
APASS_CACHE_FILE = gaia_dir / "apass_cache.pkl"
SIMBAD_CACHE_FILE = gaia_dir / "simbad_cache.pkl"
NAME_CACHE_FILE = gaia_dir / "gaia_name_cache.pkl"

if GAIA_CACHE_FILE.exists():
    with open(GAIA_CACHE_FILE, "rb") as f:
        gaia_cache = pickle.load(f)
else:
    gaia_cache = {}

if APASS_CACHE_FILE.exists():
    with open(APASS_CACHE_FILE, "rb") as f:
        apass_cache = pickle.load(f)
else:
    apass_cache = {}

# SIMBAD results cache, keyed by rounded plate-center (ra, dec), exactly
# like gaia_cache/apass_cache above. Previously, SIMBAD was queried ONCE
# PER DETECTED SOURCE (get_simbad_names looped and fired a separate
# network request for every single source on a plate) -- for a 100-source
# plate that's 100 blocking HTTP round-trips just for names, and it was
# the single biggest cost in full_scan(). Now it's one cone-search query
# per plate pointing (see get_plate_catalog_simbad below), crossmatched
# locally against every detected source, with the same on-disk cache
# reuse benefit for repeated pointings that Gaia/APASS already get.
if SIMBAD_CACHE_FILE.exists():
    with open(SIMBAD_CACHE_FILE, "rb") as f:
        simbad_cache = pickle.load(f)
else:
    simbad_cache = {}

if NAME_CACHE_FILE.exists():
    with open(NAME_CACHE_FILE, "rb") as f:
        gaia_name_cache = pickle.load(f)
else:
    gaia_name_cache = {}

# Dataset-wide reference points for the plate-limiting-magnitude quality
# signal (from Cell 1's plate_limit_lookup, itself sourced from DASCH's
# own lim_mag_apass/lim_mag_atlas in plate_manifest.csv. No network
# calls, just a CSV that's already been read). A plate scores as "deep"
# on this axis if its limiting magnitude is at or above the dataset
# median for whichever catalog it has a value for.
_apass_lim_vals = [v["lim_mag_apass"] for v in plate_limit_lookup.values() if v.get("lim_mag_apass") is not None]
_atlas_lim_vals = [v["lim_mag_atlas"] for v in plate_limit_lookup.values() if v.get("lim_mag_atlas") is not None]
_lim_mag_median_apass = float(np.median(_apass_lim_vals)) if _apass_lim_vals else None
_lim_mag_median_atlas = float(np.median(_atlas_lim_vals)) if _atlas_lim_vals else None
print(f"Plate-limit reference: median lim_mag_apass={_lim_mag_median_apass}, "
      f"median lim_mag_atlas={_lim_mag_median_atlas} "
      f"(from {len(_apass_lim_vals)} / {len(_atlas_lim_vals)} plates with data)")

# Paradigm Plate Workflow
PARADIGM_DB_FILE = gaia_dir / "paradigm_db.pkl"
PARADIGM_VERSION_FILE = gaia_dir / "paradigm_version.pkl"

if PARADIGM_DB_FILE.exists():
    with open(PARADIGM_DB_FILE, "rb") as f:
        paradigm_db = pickle.load(f)
else:
    paradigm_db = {} # { plate_path_str: [ {ra, dec, label, source}, ... ] }

paradigm_version = pickle.load(open(PARADIGM_VERSION_FILE, "rb")) if PARADIGM_VERSION_FILE.exists() else 0

PARADIGM_MATCH_SEP_ARCSEC = 3.0 # tolerance for matching paradigm objects onto other plates
PARADIGM_CLICK_PX = 15 # how close a click must be to an existing marker to select/remove it

def save_paradigm_db():
    with open(PARADIGM_DB_FILE, "wb") as f:
        pickle.dump(paradigm_db, f)

def bump_paradigm_version():
    global paradigm_version
    paradigm_version += 1
    with open(PARADIGM_VERSION_FILE, "wb") as vf:
        pickle.dump(paradigm_version, vf)

def suggest_name_at(ra, dec):
    """Look up SIMBAD/Gaia at a sky position, return a best-guess label.
    Used only for the single-point paradigm-labeling lookup (one click =
    one query is fine here), not for bulk full_scan() processing."""
    coord = SkyCoord(ra * u.deg, dec * u.deg)
    try:
        simbad = Simbad()
        simbad.TIMEOUT = 30
        result = simbad.query_region(coord, radius=PARADIGM_MATCH_SEP_ARCSEC * u.arcsec)
        if result is not None and len(result) > 0:
            name = str(result[0]["MAIN_ID"])
            if not name.startswith("Gaia"):
                return name, "SIMBAD"
    except Exception as e:
        print("[SIMBAD lookup]", e)

    try:
        job = Gaia.launch_job_async(f"""
            SELECT TOP 1 source_id, DISTANCE(
                POINT('ICRS', ra, dec),
                POINT('ICRS', {ra}, {dec})
            ) AS dist
            FROM gaiadr3.gaia_source
            WHERE CONTAINS(
                POINT('ICRS', ra, dec),
                CIRCLE('ICRS', {ra}, {dec}, {(PARADIGM_MATCH_SEP_ARCSEC * u.arcsec).to(u.deg).value})
            ) = 1
            ORDER BY dist ASC
        """)
        result = job.get_results()
        if result is not None and len(result) > 0:
            return f"Gaia {result[0]['source_id']}", "Gaia"
    except Exception as e:
        print("[Gaia lookup]", e)

    return "Unknown", "none"


# Display mode : "Paradigm" was removed as a label mode (Object ID already
# folds in paradigm-resolved names with priority); whether a plate was used
# as a paradigm reference is shown by the read-only "Reference Plate"
# checkbox instead.
display_mode = widgets.ToggleButtons(
    options=["GAIA ID", "Object ID"],
    value="GAIA ID",
    description="Labels:"
)

# Error toggle : single show/hide
error_toggle = widgets.ToggleButtons(
    options=["Show Errors", "Hide Errors"],
    value="Show Errors",
    description="Errors:"
)

# Read-only indicator : ticked whenever the currently loaded plate has any
# saved paradigm labels (i.e. it has been used as a reference plate).
paradigm_ref_indicator = widgets.Checkbox(
    value=False,
    description="Reference Plate (used for Paradigm labels)",
    disabled=True,
    indent=False
)

# Metadata
def get_plate_date(fits_path):
    try:
        header = astrofits.getheader(fits_path)
        for key in ["DATE-OBS", "DATE", "DATEOBS", "MJD-OBS"]:
            if key in header:
                return str(header[key])
        return "Unknown"
    except:
        return "Unknown"

# Error detection
def detect_plate_errors(data):

    errors = {
        "scratches":  [],
        "trailing":   [],
        "saturation": [],
        "dust":       [],
        "edge":       False,
        "dead_zone_fraction": 0.0,
        "saturation_area_fraction": 0.0,
    }

    h, w = data.shape

    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    if hi == lo:
        return errors
    norm = np.clip((data - lo) / (hi - lo), 0, 1)

    # Scratches. A relative-to-max threshold alone will almost always
    # return SOME "peaks" even from pure grain-noise edges, since it's
    # scaled to whatever this specific frame's own maximum happens to be.
    # Fixed by ALSO requiring each accepted peak to be a genuine
    # statistical outlier in the Hough accumulator (well above the noise
    # floor of the whole accumulator) -- a true straight scratch
    # concentrates edge energy along one line and produces a peak far
    # above that floor; random edge noise essentially never does.
    try:
        scale   = 6
        small   = ndimage.zoom(norm, 1 / scale, order=1)
        sobel_h = ndimage.sobel(small, axis=0)
        sobel_v = ndimage.sobel(small, axis=1)
        edges   = np.hypot(sobel_h, sobel_v)
        edges   = (edges > np.percentile(edges, 97)).astype(np.uint8)

        tested_angles = np.linspace(-np.pi / 2, np.pi / 2, 90, endpoint=False)
        hspace, angles, dists = hough_line(edges, theta=tested_angles)

        peaks = hough_line_peaks(
            hspace, angles, dists,
            num_peaks=8,
            min_distance=20,
            threshold=0.35 * hspace.max()
        )

        hspace_mean = float(np.mean(hspace))
        hspace_std  = float(np.std(hspace))
        sig_thresh  = hspace_mean + 6 * hspace_std

        sh, sw = small.shape
        for peak_val, angle, dist in zip(*peaks):
            if peak_val < sig_thresh:
                continue

            cos_a, sin_a = np.cos(angle), np.sin(angle)
            if abs(sin_a) > 1e-6:
                x0_s, x1_s = 0, sw - 1
                y0_s = (dist - x0_s * cos_a) / sin_a
                y1_s = (dist - x1_s * cos_a) / sin_a
            else:
                y0_s, y1_s = 0, sh - 1
                x0_s = x1_s = dist / cos_a if abs(cos_a) > 1e-6 else 0

            y0_s = float(np.clip(y0_s, 0, sh - 1))
            y1_s = float(np.clip(y1_s, 0, sh - 1))
            x0_s = float(np.clip(x0_s, 0, sw - 1))
            x1_s = float(np.clip(x1_s, 0, sw - 1))

            errors["scratches"].append({
                "x0": x0_s * scale, "y0": y0_s * scale,
                "x1": x1_s * scale, "y1": y1_s * scale,
            })
    except Exception as e:
        print(f"[SCRATCH DETECT] {e}")

    # Trailing stars (only flags sources with extreme elongation (>5) and minimum area)
    try:
        bright_mask = (norm > np.percentile(norm, 95)).astype(np.uint8)
        labeled, n_obj = ndimage.label(bright_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 40 or area > 0.005 * h * w:
                continue

            coords = np.argwhere(region)
            if len(coords) < 10:
                continue

            cov  = np.cov(coords[:, 1], coords[:, 0])
            eigs = np.linalg.eigvalsh(cov)
            eigs = np.sort(np.abs(eigs))
            if eigs[0] < 1e-6:
                continue

            elongation = np.sqrt(eigs[1] / eigs[0])
            if elongation > 5.0:
                cy_r, cx_r = coords.mean(axis=0)
                angle = 0.5 * np.degrees(np.arctan2(
                    2 * cov[0, 1], cov[0, 0] - cov[1, 1]
                ))
                major = 2 * np.sqrt(eigs[1])
                minor = 2 * np.sqrt(eigs[0])
                errors["trailing"].append({
                    "x": float(cx_r), "y": float(cy_r),
                    "w": float(major), "h": float(minor),
                    "angle": float(angle)
                })
    except Exception as e:
        print(f"[TRAIL DETECT] {e}")

    # Saturation / blooming (must be large and at the absolute ceiling).
    # Also tracks the TOTAL saturated area as a fraction of the frame,
    # not just the count of distinct blobs -- one dominant overexposed
    # star is a single connected component and would never trip a
    # count-based check, no matter how much of the frame it consumes.
    try:
        sat_thresh = np.percentile(norm, 99.9)
        sat_mask   = (norm >= sat_thresh).astype(np.uint8)
        sat_mask   = ndimage.binary_dilation(sat_mask, iterations=2).astype(np.uint8)

        errors["saturation_area_fraction"] = float(sat_mask.sum()) / float(h * w)

        labeled, n_obj = ndimage.label(sat_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 200:
                continue

            coords  = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:, 0].max() - coords[:, 0].min()) / 2
            rx = (coords[:, 1].max() - coords[:, 1].min()) / 2
            r  = np.hypot(rx, ry)

            errors["saturation"].append({
                "x": float(cx_r), "y": float(cy_r), "r": float(r)
            })
    except Exception as e:
        print(f"[SAT DETECT] {e}")

    # Dust spots / emulsion streaks. Genuine dust damage is a LOCAL
    # anomaly -- noticeably darker than its immediate surroundings -- not
    # just "somewhere in the bottom 5% of pixel values across the whole
    # frame". Fixed by working on the RESIDUAL from a smoothed local
    # background (so only genuine localized dark anomalies survive) and
    # thresholding in units of a robust sigma estimate (MAD) instead of a
    # fixed global percentile.
    try:
        bg_size = max(15, min(h, w) // 12)
        local_bg = ndimage.uniform_filter(norm, size=bg_size)
        residual = norm - local_bg

        med_resid = float(np.nanmedian(residual))
        mad = float(np.nanmedian(np.abs(residual - med_resid)))
        sigma_est = max(1.4826 * mad, 1e-3)  # robust sigma, floored to avoid div-by-~0

        dark_mask = (residual <= (med_resid - 5 * sigma_est)).astype(np.uint8)
        dark_mask = ndimage.binary_opening(dark_mask, iterations=1).astype(np.uint8)
        labeled, n_obj = ndimage.label(dark_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 15 or area > 0.02 * h * w:
                continue

            coords  = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:, 0].max() - coords[:, 0].min()) / 2
            rx = (coords[:, 1].max() - coords[:, 1].min()) / 2

            if rx < 1e-6 or ry < 1e-6:
                continue

            r = np.hypot(rx, ry)
            errors["dust"].append({
                "x": float(cx_r), "y": float(cy_r), "r": float(r)
            })
    except Exception as e:
        print(f"[DUST DETECT] {e}")

    # Dead zone / vignetting : some cutouts include a region where the plate
    # emulsion doesn't cover part of the frame (vignetting cutoff, a
    # stitching edge, or the cutout extending past the plate boundary).
    # Unlike the border-only edge check below, this can eat a large
    # diagonal wedge out of the MIDDLE of the frame. Detected as the
    # largest connected blob of pixels sitting near the raw
    # (non-normalized) global minimum.
    try:
        raw_min = np.nanmin(data)
        raw_max = np.nanmax(data)
        tol = max(1.0, 0.02 * (raw_max - raw_min))
        dead_mask = (data <= raw_min + tol).astype(np.uint8)
        labeled_dead, n_dead_obj = ndimage.label(dead_mask)
        largest_dead_area = 0
        if n_dead_obj > 0:
            sizes = ndimage.sum(dead_mask, labeled_dead, index=range(1, n_dead_obj + 1))
            if len(sizes):
                largest_dead_area = int(np.max(sizes))
        errors["dead_zone_fraction"] = float(largest_dead_area) / float(h * w)
    except Exception as e:
        print(f"[DEAD ZONE DETECT] {e}")
        errors["dead_zone_fraction"] = 0.0

    # Edge artifacts
    try:
        border = max(20, int(min(h, w) * 0.05))
        center_med = np.nanmedian(norm[border:h-border, border:w-border])
        center_std = np.nanstd(norm[border:h-border, border:w-border])

        edge_strips = [
            norm[:border, :],
            norm[h-border:, :],
            norm[:, :border],
            norm[:, w-border:],
        ]

        for strip in edge_strips:
            strip_med = np.nanmedian(strip)
            if abs(strip_med - center_med) > 3 * center_std:
                errors["edge"] = True
                break
    except Exception as e:
        print(f"[EDGE DETECT] {e}")

    return errors


def annotate_errors(ax, errors):
    
    # Draw all error types at once, each with its own color and category label
    legend_handles = []

    # Scratches
    for s in errors["scratches"]:
        ax.plot(
            [s["x0"], s["x1"]], [s["y0"], s["y1"]],
            color="magenta", linewidth=1.2, alpha=0.8, linestyle="--"
        )
    if errors["scratches"]:
        legend_handles.append(
            Line2D([0], [0], color="magenta", linestyle="--", label=f'Scratch ×{len(errors["scratches"])}')
        )

    # Trailing stars
    for t in errors["trailing"]:
        ellipse = mpatches.Ellipse(
            (t["x"], t["y"]),
            width=t["w"], height=t["h"],
            angle=t["angle"],
            edgecolor="orange", facecolor="none",
            linewidth=1.5, alpha=0.85
        )
        ax.add_patch(ellipse)
        ax.text(
            t["x"] + t["w"] / 2 + 3, t["y"],
            "trail", color="orange", fontsize=5,
            va="center"
        )
    if errors["trailing"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="orange", facecolor="none",
                           label=f'Trailing ×{len(errors["trailing"])}')
        )

    # Saturation
    for s in errors["saturation"]:
        circ = plt.Circle(
            (s["x"], s["y"]), s["r"],
            edgecolor="red", facecolor="none",
            linewidth=1.5, alpha=0.8, linestyle="-."
        )
        ax.add_patch(circ)
        ax.text(
            s["x"], s["y"] - s["r"] - 4,
            "sat", color="red", fontsize=5,
            ha="center"
        )
    if errors["saturation"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="red", facecolor="none",
                           label=f'Saturation ×{len(errors["saturation"])}')
        )

    sat_frac = errors.get("saturation_area_fraction", 0.0)
    if sat_frac >= 0.05:
        legend_handles.append(
            mpatches.Patch(edgecolor="red", facecolor="red", alpha=0.3,
                           label=f"Sat area {sat_frac * 100:.0f}%")
        )

    # Dust spots
    for d in errors["dust"]:
        circ = plt.Circle(
            (d["x"], d["y"]), d["r"],
            edgecolor="deepskyblue", facecolor="none",
            linewidth=1.2, alpha=0.8, linestyle=":"
        )
        ax.add_patch(circ)
        ax.text(
            d["x"], d["y"] - d["r"] - 4,
            "dust", color="deepskyblue", fontsize=5,
            ha="center"
        )
    if errors["dust"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="deepskyblue", facecolor="none",
                           label=f'Dust ×{len(errors["dust"])}')
        )

    # Dead zone / vignetting
    dead_frac = errors.get("dead_zone_fraction", 0.0)
    if dead_frac >= 0.05:
        legend_handles.append(
            mpatches.Patch(edgecolor="gray", facecolor="gray", alpha=0.5,
                           label=f"Dead zone {dead_frac * 100:.0f}%")
        )

    # Edge artifacts
    if errors["edge"]:
        ax_h, ax_w = ax.get_ylim(), ax.get_xlim()
        img_h = abs(ax_h[1] - ax_h[0])
        img_w = abs(ax_w[1] - ax_w[0])
        rect = mpatches.Rectangle(
            (min(ax_w), min(ax_h)), img_w, img_h,
            edgecolor="lime", facecolor="none",
            linewidth=2.5, alpha=0.7
        )
        ax.add_patch(rect)
        legend_handles.append(
            mpatches.Patch(edgecolor="lime", facecolor="none",
                           label="Edge artifact")
        )

    if legend_handles:
        ax.legend(
            handles=legend_handles,
            loc="upper left",
            fontsize=6,
            framealpha=0.6,
            facecolor="black",
            labelcolor="white",
            edgecolor="gray"
        )

# Plate quality categorization: too many errors to use / matches the catalog well / defective altogether / fair
def classify_plate_quality(errors, n_sources, n_matched=None, lim_mag_apass=None, lim_mag_atlas=None):

    n_scratches  = len(errors["scratches"])
    n_trailing   = len(errors["trailing"])
    n_saturation = len(errors["saturation"])
    n_dust       = len(errors["dust"])
    n_edge       = 1 if errors["edge"] else 0
    dead_zone_fraction = errors.get("dead_zone_fraction", 0.0)
    saturation_area_fraction = errors.get("saturation_area_fraction", 0.0)

    total_defects = n_scratches + n_trailing + n_saturation + n_dust + n_edge

    # Hard downgrade: a large dead zone / vignetting cutoff makes a big
    # chunk of the frame unusable for photometry no matter how clean the
    # rest of the checks look, or how good its catalog match / limiting
    # magnitude are.
    if dead_zone_fraction >= 0.08:
        return "defective"

    # Hard downgrade: a dominant overexposed/blooming region -- even if
    # it's just ONE connected blob -- consuming a large fraction of the
    # frame is just as disqualifying as many small saturated spots.
    if saturation_area_fraction >= 0.05:
        return "defective"

    # Defective: the plate itself looks unusable (heavy blooming, edge issues
    # combined with other defects, or almost no real detections on a messy frame)
    if (n_saturation >= 8) or (errors["edge"] and total_defects >= 6) or (n_sources < 3 and total_defects >= 3):
        return "defective"

    # Too many errors: salvageable in principle but heavily contaminated
    if total_defects >= 8:
        return "too_many_errors"

    # Defect-based "clean" signal. During prescan(), n_matched is None (no
    # Gaia/APASS crossmatch has happened yet), so this falls back to just
    # the defect count; during full_scan(), the catalog match fraction is
    # folded in too, refining the call.
    defect_based_good = total_defects < 8
    if n_matched is not None:
        match_fraction = (n_matched / n_sources) if n_sources > 0 else 0.0
        defect_based_good = defect_based_good and (match_fraction >= 0.5)

    # Plate-limit signal (DASCH's own lim_mag_apass/lim_mag_atlas from the
    # manifest, no network calls): deeper (numerically larger) limiting
    # magnitude means the plate reaches fainter stars, so generally a
    # cleaner, better-exposed plate. Compared against the dataset-wide
    # median computed above. If neither value is available for this
    # plate, treated as a neutral pass so plates without manifest coverage
    # aren't unfairly excluded.
    lim_mag = lim_mag_apass if lim_mag_apass is not None else lim_mag_atlas
    lim_mag_reference = _lim_mag_median_apass if lim_mag_apass is not None else _lim_mag_median_atlas
    lim_mag_is_deep = (lim_mag is None) or (lim_mag_reference is None) or (lim_mag >= lim_mag_reference)

    # "good_match" requires BOTH signals to agree. A scratchy/low-match
    # plate never qualifies just because it's deep, and a defect-free but
    # shallow plate never qualifies just because it looks clean.
    if defect_based_good and lim_mag_is_deep:
        return "good_match"

    return "fair"

# Retroactive label enrichment : re-resolve names for all already-scanned plates using whatever is now in gaia_name_cache (without re-querying Gaia or SIMBAD).
def enrich_previous_plates(plate_db, up_to_key):

    keys = list(plate_db.keys())
    stop = keys.index(up_to_key) if up_to_key in keys else len(keys)

    for f in keys[:stop]:

        r = plate_db[f].get("render")

        if r is None or "gaia_names" not in r:
            continue

        old_resolved = r["resolved_names"]
        new_resolved = resolve_names(r["gaia_names"], ["Unknown"] * len(r["gaia_names"]))

        merged = []
        for old, new in zip(old_resolved, new_resolved):
            if old == "Unknown" and new != "Unknown":
                merged.append(new)
            else:
                merged.append(resolve_gaia_label(old))

        r["resolved_names"] = merged

# GAIA catalog (used only to assign GAIA source IDs to detected sources)
def get_plate_catalog_gaia(wcs, shape):

    cx, cy = shape[1] / 2, shape[0] / 2
    ra_c, dec_c = wcs.pixel_to_world_values(cx, cy)

    ra_c = float(np.array(ra_c))
    dec_c = float(np.array(dec_c))

    key = (round(ra_c, 4), round(dec_c, 4))

    if key in gaia_cache:
        return gaia_cache[key]

    radius = 0.15 * u.deg
    center = SkyCoord(ra_c * u.deg, dec_c * u.deg)

    query = f"""
    SELECT source_id, ra, dec
    FROM gaiadr3.gaia_source
    WHERE CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra_c}, {dec_c}, {radius.to(u.deg).value})
    ) = 1
    """

    try:
        job = Gaia.launch_job_async(query)
        result = job.get_results()

        if result is None:
            result = []

        gaia_cache[key] = result

        with open(GAIA_CACHE_FILE, "wb") as f:
            pickle.dump(gaia_cache, f)

        print(f"[GAIA] loaded {len(result)} sources")

        return result

    except Exception as e:
        print("[GAIA ERROR]", e)
        gaia_cache[key] = []
        return []

# APASS catalog (used for B-band calibration magnitudes, via VizieR II/336/apass9)
def get_plate_catalog_apass(wcs, shape):

    cx, cy = shape[1] / 2, shape[0] / 2
    ra_c, dec_c = wcs.pixel_to_world_values(cx, cy)

    ra_c = float(np.array(ra_c))
    dec_c = float(np.array(dec_c))

    key = (round(ra_c, 4), round(dec_c, 4))

    if key in apass_cache:
        return apass_cache[key]

    radius = 0.15 * u.deg
    center = SkyCoord(ra_c * u.deg, dec_c * u.deg)

    try:
        vquery = Vizier(
            columns=["recno", "RAJ2000", "DEJ2000", "Vmag", "e_Vmag", "Bmag", "e_Bmag"],
            row_limit=-1
        )

        result_list = vquery.query_region(
            center,
            radius=radius,
            catalog="II/336/apass9"
        )

        if result_list is None or len(result_list) == 0:
            apass_cache[key] = []
            return []

        result = result_list[0]
        result = result[np.isfinite(result["Bmag"])]

        result.rename_column("RAJ2000", "ra")
        result.rename_column("DEJ2000", "dec")
        result.rename_column("Bmag", "b_mag")

        apass_cache[key] = result

        with open(APASS_CACHE_FILE, "wb") as f:
            pickle.dump(apass_cache, f)

        print(f"[APASS] loaded {len(result)} sources")

        return result

    except Exception as e:
        print("[APASS ERROR]", e)
        apass_cache[key] = []
        return []

# SIMBAD catalog for a whole plate, ONE cone search at the plate center
# (same radius/caching pattern as Gaia/APASS above) instead of one query
# PER DETECTED SOURCE. This is the fix for full_scan() being slow -- the
# old get_simbad_names() looped and issued a separate network request for
# every single source on a plate.
def get_plate_catalog_simbad(wcs, shape):

    cx, cy = shape[1] / 2, shape[0] / 2
    ra_c, dec_c = wcs.pixel_to_world_values(cx, cy)

    ra_c = float(np.array(ra_c))
    dec_c = float(np.array(dec_c))

    key = (round(ra_c, 4), round(dec_c, 4))

    if key in simbad_cache:
        return simbad_cache[key]

    radius = 0.15 * u.deg
    center = SkyCoord(ra_c * u.deg, dec_c * u.deg)

    try:
        simbad = Simbad()
        simbad.TIMEOUT = 60
        simbad.add_votable_fields("main_id", "ra", "dec")

        result = simbad.query_region(center, radius=radius)

        if result is None:
            result = []

        simbad_cache[key] = result

        with open(SIMBAD_CACHE_FILE, "wb") as f:
            pickle.dump(simbad_cache, f)

        print(f"[SIMBAD] loaded {len(result)} sources")

        return result

    except Exception as e:
        print("[SIMBAD ERROR]", e)
        simbad_cache[key] = []
        return []

# Crossmatch detected sources against the whole-plate SIMBAD catalog above
# (local, no network) -- mirrors match_detected_sources_gaia's approach.
def match_detected_sources_simbad(ra, dec, catalog, max_sep_arcsec=5):

    if catalog is None or len(catalog) == 0:
        return ["Unknown"] * len(ra)

    try:
        # SIMBAD's default RA/DEC votable fields are sexagesimal strings
        # (hours:min:sec / deg:min:sec), hence the (hourangle, deg) units.
        cat_coords = SkyCoord(catalog["RA"], catalog["DEC"], unit=(u.hourangle, u.deg))
    except Exception as e:
        print("[SIMBAD MATCH]", e)
        return ["Unknown"] * len(ra)

    src_coords = SkyCoord(ra * u.deg, dec * u.deg)
    idx, sep, _ = match_coordinates_sky(src_coords, cat_coords)

    labels = []
    for j in range(len(src_coords)):
        if sep[j].arcsec <= max_sep_arcsec:
            name = str(catalog["MAIN_ID"][idx[j]])
            labels.append("Unknown" if name.startswith("Gaia") else name)
        else:
            labels.append("Unknown")

    return labels

# GAIA Crossmatch labels
def match_detected_sources_gaia(ra, dec, catalog, max_sep_arcsec=2.5):

    if catalog is None or len(catalog) == 0:
        return ["Unknown"] * len(ra)

    cat_coords = SkyCoord(catalog["ra"], catalog["dec"], unit="deg")
    src_coords = SkyCoord(ra * u.deg, dec * u.deg)

    idx, sep, _ = match_coordinates_sky(src_coords, cat_coords)

    labels = []

    for j in range(len(src_coords)):
        if sep[j].arcsec <= max_sep_arcsec and sep[j].arcsec < 3.0:
            labels.append(f"Gaia {catalog['source_id'][idx[j]]}")
        else:
            labels.append("Unknown")

    return labels

# Paradigm crossmatch labels : pools every labeled object across all paradigm plates
def match_against_paradigm(ra, dec, max_sep_arcsec=PARADIGM_MATCH_SEP_ARCSEC):

    all_entries = [e for entries in paradigm_db.values() for e in entries]
    if not all_entries:
        return ["Unknown"] * len(ra)

    cat_coords = SkyCoord([e["ra"] for e in all_entries], [e["dec"] for e in all_entries], unit="deg")
    src_coords = SkyCoord(ra * u.deg, dec * u.deg)
    idx, sep, _ = match_coordinates_sky(src_coords, cat_coords)

    labels = []
    for j in range(len(src_coords)):
        if sep[j].arcsec <= max_sep_arcsec:
            labels.append(all_entries[idx[j]]["label"])
        else:
            labels.append("Unknown")
    return labels

def update_name_cache(gaia_labels, simbad_names):

    changed = False

    for g, s in zip(gaia_labels, simbad_names):

        if s == "Unknown":
            continue

        if not g.startswith("Gaia "):
            continue

        try:
            source_id = int(g.replace("Gaia ", ""))

            gaia_name_cache[source_id] = s
            changed = True

        except:
            pass

    if changed:
        with open(NAME_CACHE_FILE, "wb") as f:
            pickle.dump(gaia_name_cache, f)


def resolve_names(gaia_labels, simbad_names):

    labels = []

    for g, s in zip(gaia_labels, simbad_names):

        if s != "Unknown":
            labels.append(s)
            continue

        if g.startswith("Gaia "):

            try:
                source_id = int(g.replace("Gaia ", ""))

                if source_id in gaia_name_cache:
                    labels.append(
                        gaia_name_cache[source_id]
                    )
                    continue

            except:
                pass

        labels.append("Unknown")

    return labels

def resolve_gaia_label(label):
    """If `label` is a raw 'Gaia <id>' string, try to resolve it through
    the name cache and return the real name instead. Falls back to the raw
    Gaia ID string only if no cached name exists yet."""
    if isinstance(label, str) and label.startswith("Gaia "):
        try:
            source_id = int(label.replace("Gaia ", "").strip())
            if source_id in gaia_name_cache:
                return gaia_name_cache[source_id]
        except:
            pass
    return label

# R CrB Match
def find_target_match(ra, dec, max_sep_arcsec=10):

    src = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
    sep = src.separation(TARGET_STAR)

    idx = np.argmin(sep)
    best = sep[idx].arcsec

    return (best < max_sep_arcsec), idx, best

def detect_target(wcs, data, ra, dec):
    """Match detected sources against R CrB's known sky position, with a
    fallback for badly overexposed/blooming target stars that DAOStarFinder
    fails to centroid as a normal point source. If no detected source
    matches within the normal tolerance, this computes R CrB's expected
    pixel position directly from the WCS and checks the raw pixel data
    there for a bright/saturated blob."""
    matched_found, idx, sep = find_target_match(ra, dec)
    has_target = matched_found
    target_fallback = None

    if not matched_found:
        try:
            tx, ty = wcs.world_to_pixel_values(TARGET_STAR.ra.deg, TARGET_STAR.dec.deg)
            tx, ty = float(np.array(tx)), float(np.array(ty))
            margin = 5
            if (-margin <= tx < data.shape[1] + margin) and (-margin <= ty < data.shape[0] + margin):
                search_r = 25
                x0 = max(int(tx - search_r), 0)
                x1 = min(int(tx + search_r) + 1, data.shape[1])
                y0 = max(int(ty - search_r), 0)
                y1 = min(int(ty + search_r) + 1, data.shape[0])
                if x1 > x0 and y1 > y0:
                    local = data[y0:y1, x0:x1]
                    local_thresh = np.nanpercentile(data, 97)
                    bright_mask_local = local > local_thresh
                    if bright_mask_local.any():
                        ys_idx, xs_idx = np.nonzero(bright_mask_local)
                        weights = local[ys_idx, xs_idx]
                        cx_local = np.average(xs_idx, weights=weights)
                        cy_local = np.average(ys_idx, weights=weights)
                        target_fallback = {"x": float(x0 + cx_local), "y": float(y0 + cy_local)}
                        has_target = True
        except Exception as e:
            print(f"[TARGET FALLBACK] {e}")

    return matched_found, idx, sep, has_target, target_fallback

# Plate database : loaded once; populated/refreshed by prescan()/full_scan()
if PLATE_DB_FILE.exists():
    print("Loading saved plate database...")
    with open(PLATE_DB_FILE, "rb") as fp:
        plate_db = pickle.load(fp)
else:
    plate_db = {}

scan_progress = widgets.IntProgress(value=0, min=0, max=max(len(cutouts), 1), description="Scanning:")
scan_status = widgets.Label(value=f"Loaded {len(plate_db)} cached plates" if plate_db else "Not scanned yet")

# Progress dashboard (percentage bar, plates/sec, elapsed, ETA), same
# pattern as the download-progress dashboard in 01_Load_Cutout.ipynb's
# Cell 1. Shared by both prescan() and full_scan(). scan_progress (the
# IntProgress bar above) is kept around and still updated for internal
# bookkeeping, but no longer displayed -- this HTML dashboard already
# renders its own bar plus %, rate, elapsed, and ETA, so showing both was
# redundant.
scan_progress_html = widgets.HTML(value="")

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:  # NaN/negative guard
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def render_scan_progress(n_done, total, start_time, status_word="Scanning", color="#4CAF50"):
    elapsed = time.monotonic() - start_time
    pct = (n_done / total * 100) if total else 0
    rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
    remaining = ((total - n_done) / rate) if rate > 0 else None

    bar_width = 380
    filled = int(bar_width * pct / 100)

    scan_progress_html.value = f"""
    <div style="font-family: monospace; font-size: 13px; line-height: 1.5;">
      <div style="display:flex; align-items:center;">
        <div style="width:{bar_width}px; height:16px; background:#333;
                    border-radius:4px; overflow:hidden; margin-right:10px;">
          <div style="width:{filled}px; height:100%; background:{color};
                      transition: width 0.3s;"></div>
        </div>
        <b>{pct:5.1f}%</b>
      </div>
      <div style="margin-top:4px;">
        {status_word} &nbsp;
        <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
        <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
        Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp;
        ETA <b>{format_eta(remaining)}</b>
      </div>
    </div>
    """

def paradigm_established():
    """True once at least one object has been labeled on at least one
    paradigm plate. Gates full_scan() (via 'Apply Labels') and the main
    viewer's dropdown further down."""
    return any(len(v) > 0 for v in paradigm_db.values())

_full_scan_has_run = False

def _prescan_one_plate(f):
    """Runs entirely offline (no network, no shared-state mutation) so it's
    safe to call from multiple threads at once. Returns (f, action,
    payload) describing what the main thread should do with plate_db;
    plate_db itself is only ever written back on the main thread, so
    there's no race condition even though many of these run concurrently.
    action is one of: "skip", "light_refresh", "new".
    """
    cached = plate_db.get(str(f))
    has_render = cached is not None and cached.get("render") is not None

    needs_target_refresh = (
        has_render
        and cached["render"].get("target_version", -1) != TARGET_DETECTION_VERSION
    )
    needs_quality_refresh = (
        has_render
        and cached["render"].get("quality_version", -1) != QUALITY_VERSION
    )
    needs_light_refresh = needs_target_refresh or needs_quality_refresh

    if has_render and not needs_light_refresh:
        return (f, "skip", None)

    if needs_light_refresh:
        r = cached["render"]
        payload = {
            "quality": None, "quality_version": None, "errors": None, "phase": None,
            "found": None, "target_idx": None, "target_fallback": None,
            "target_version": None, "target": None,
        }

        if needs_quality_refresh:
            try:
                data_for_quality = astrofits.getdata(f).astype(float)
                data_for_quality = np.nan_to_num(data_for_quality, nan=np.nanmedian(data_for_quality))
                new_errors = detect_plate_errors(data_for_quality)

                phase = "full" if "gaia_names" in r else r.get("phase", "prescan")

                if phase == "full":
                    gaia_names_existing = r.get("gaia_names", [])
                    catalog_b_existing = r.get("catalog_b", np.full(len(gaia_names_existing), np.nan))
                    matched_mask = (
                        np.array([g != "Unknown" for g in gaia_names_existing])
                        | np.isfinite(catalog_b_existing)
                    )
                    n_matched = int(np.sum(matched_mask))
                else:
                    n_matched = None

                lim_mag_apass, lim_mag_atlas = get_plate_limits(f)
                quality = classify_plate_quality(
                    new_errors, cached["n"], n_matched, lim_mag_apass, lim_mag_atlas
                )
                payload["errors"] = new_errors
                payload["phase"] = phase
                payload["quality"] = quality
                payload["quality_version"] = QUALITY_VERSION
            except Exception as e:
                payload["quality_error"] = str(e)

        if needs_target_refresh:
            try:
                hdr = astrofits.getheader(f)
                wcs = WCS(hdr)
                ra, dec = wcs.pixel_to_world_values(r["sources"][r["x_col"]], r["sources"][r["y_col"]])
                ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)
                data_for_target = astrofits.getdata(f).astype(float)
                data_for_target = np.nan_to_num(data_for_target, nan=np.nanmedian(data_for_target))
                matched_found, idx, sep, has_target, target_fallback = detect_target(
                    wcs, data_for_target, ra, dec
                )
                payload["found"] = matched_found
                payload["target_idx"] = idx
                payload["target_fallback"] = target_fallback
                payload["target_version"] = TARGET_DETECTION_VERSION
                payload["target"] = has_target
            except Exception as e:
                payload["target_error"] = str(e)

        return (f, "light_refresh", payload)

    # Brand new / never-cached plate
    try:
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(f)

        tag = "error"
        quality = "defective"
        n = 0
        has_target = False
        cached_render = None

        if sources is not None and len(sources) > 0:
            n = len(sources)

            hdr = astrofits.getheader(f)
            wcs = WCS(hdr)

            ra, dec = wcs.pixel_to_world_values(sources[x_col], sources[y_col])
            ra = np.array(ra, dtype=float)
            dec = np.array(dec, dtype=float)

            matched_found, idx, sep, has_target, target_fallback = detect_target(wcs, data, ra, dec)
            found = matched_found

            sharp = sources["sharpness"] if "sharpness" in sources.colnames else None
            elong = sources["elongation"] if "elongation" in sources.colnames else None

            bad = 0
            if sharp is not None:
                bad += np.sum(np.array(sharp) < 0.1)
            if elong is not None:
                bad += np.sum(np.array(elong) > 3)
            bad_frac = bad / n

            if bad_frac > 0.5:
                tag = "messy"
            elif n < 10:
                tag = "empty"
            elif n < 100:
                tag = "crowded"
            else:
                tag = "rich"

            plate_errors = detect_plate_errors(data)

            lim_mag_apass, lim_mag_atlas = get_plate_limits(f)
            quality = classify_plate_quality(
                plate_errors, n, None, lim_mag_apass, lim_mag_atlas
            )

            cached_render = {
                "sources":         sources,
                "x_col":           x_col,
                "y_col":           y_col,
                "found":           found,
                "target_idx":      idx,
                "target_fallback": target_fallback,
                "target_version":  TARGET_DETECTION_VERSION,
                "errors":          plate_errors,
                "phase":           "prescan",
                "quality_version": QUALITY_VERSION,
            }

        entry = {
            "tag":           tag,
            "quality":       quality,
            "n":             n,
            "date":          get_plate_date(f),
            "target":        has_target,
            "human_verdict": None,
            "render":        cached_render,
        }
        return (f, "new", entry)

    except Exception as e:
        entry = {
            "tag": "error", "quality": "defective", "n": 0,
            "date": get_plate_date(f), "target": False,
            "human_verdict": None, "render": None,
            "error": str(e),
        }
        return (f, "new", entry)

DEBUG_N = 30  # Pass plates=cutouts[:DEBUG_N] to prescan() or full_scan() below to test on a subset.

def prescan(plates=None, max_workers=PRESCAN_WORKERS):
    """
    Fast, network-free pass: source detection, defect/dead-zone checks,
    target detection, and quality classification using ONLY offline data
    (detected sources + detect_plate_errors() + DASCH's own
    lim_mag_apass/lim_mag_atlas from Cell 1's plate_limit_lookup). No
    Gaia/SIMBAD/APASS network calls happen here -- those are deferred to
    full_scan(), which only runs after a paradigm plate has been labeled.

    Runs the per-plate work (_prescan_one_plate) across max_workers threads
    at once. Plates already cached with a current target_version AND
    quality_version are skipped entirely. Plates whose target/quality
    logic is stale get a cheap offline re-check.
    """
    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    scan_progress.max = total
    scan_progress.value = 0

    completed = 0
    autosave_every = 500
    start_time = time.monotonic()
    _tick_state = {"last_render": 0.0}

    def _tick(n_done):
        scan_progress.value = n_done
        now = time.monotonic()
        if now - _tick_state["last_render"] > 0.15 or n_done == total:
            render_scan_progress(n_done, total, start_time, status_word="Pre-scanning", color="#4CAF50")
            _tick_state["last_render"] = now

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_prescan_one_plate, f): f for f in target_plates}

        for fut in as_completed(futures):
            f = futures[fut]

            try:
                f_ret, action, payload = fut.result()
            except Exception as e:
                print(f"[PRESCAN WORKER ERROR] {f.name}: {e}")
                completed += 1
                _tick(completed)
                continue

            if action == "skip":
                pass

            elif action == "light_refresh":
                cached = plate_db.get(str(f))
                if cached is not None and cached.get("render") is not None:
                    r = cached["render"]
                    if payload.get("quality") is not None:
                        r["errors"] = payload["errors"]
                        r["phase"] = payload["phase"]
                        r["quality_version"] = payload["quality_version"]
                        plate_db[str(f)]["quality"] = payload["quality"]
                    if "quality_error" in payload:
                        print(f"[PRESCAN QUALITY REFRESH] {f.name}: {payload['quality_error']}")
                    if payload.get("target_version") is not None:
                        r["found"] = payload["found"]
                        r["target_idx"] = payload["target_idx"]
                        r["target_fallback"] = payload["target_fallback"]
                        r["target_version"] = payload["target_version"]
                        plate_db[str(f)]["target"] = payload["target"]
                    if "target_error" in payload:
                        print(f"[PRESCAN TARGET REFRESH] {f.name}: {payload['target_error']}")

            elif action == "new":
                plate_db[str(f)] = payload
                if "error" in payload:
                    print(f"[PRESCAN ERROR] {f.name}: {payload['error']}")

            completed += 1
            scan_status.value = f.name
            _tick(completed)

            if completed % autosave_every == 0:
                with open(PLATE_DB_FILE, "wb") as fp:
                    pickle.dump(plate_db, fp)
                print(f"Pre-scan autosaved {completed} plates")

    render_scan_progress(completed, total, start_time, status_word="Pre-scan complete", color="#4CAF50")
    scan_status.value = "Pre-scan complete"
    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)
    print(f"Pre-scanned {len(plate_db)} plates (offline pass -- no network calls, {max_workers} parallel workers)")
    refresh_filter_options()
    rebuild_paradigm_dropdown()
    rebuild_main_dropdown()
    _refresh_review_queue()



# ============================================================
# Robust per-source photometry helpers (ported from
# test_Simple_Photometry_Clayton.ipynb): centroid refinement,
# crowding-aware adaptive apertures, local background via CircularAnnulus
# with neighbor-star exclusion, saturation detection, wings-only FWHM for
# saturated sources, a self-derived 5-sigma plate limit, and a two-star
# differential-photometry cross-check against APASS. All of this runs
# inside full_scan() only (it needs the APASS catalog, a network call),
# never in prescan().
# ============================================================

CENTROID_BOX = 21              # px; must comfortably contain a star's peak, not a neighbor
DEFAULT_APERTURE_R = 40.0      # px; matches the large aperture used in the reference notebook
MIN_APERTURE_R = 8.0           # px; never shrink an aperture below this, however crowded
ANNULUS_WIDTH = 5.0            # px; width of the local-background ring outside each aperture
APERTURE_SAFETY_BUFFER = 2.0   # px; gap kept between two neighboring apertures when shrinking
SATURATION_FRACTION = 0.995    # a pixel counts as "near max" at >= this fraction of the source's own peak
SATURATION_MIN_PIXELS = 5      # a source is flagged saturated once this many near-max pixels exist
PLATE_LIMIT_NSIGMA = 5         # sigma level for the self-derived plate-limit diagnostic
FWHM_EDGE_RADII = np.arange(25)  # radial-profile bins (px) used for the wings-only FWHM fit


def refine_source_centroids(data, xs, ys, box_size=CENTROID_BOX):
    """Center-of-mass centroid refinement in a local box around each
    detected position -- corrects the few-pixel miscentering that biases
    flux captured in a fixed-position aperture. Falls back to the
    original position for any source whose refinement fails or wanders
    implausibly far (>= box_size away, i.e. probably locked onto a
    neighbor instead)."""
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    try:
        xs_ref, ys_ref = centroid_sources(
            data, xs, ys, box_size=box_size, centroid_func=centroid_com
        )
        bad = ~np.isfinite(xs_ref) | ~np.isfinite(ys_ref) | (np.hypot(xs_ref - xs, ys_ref - ys) >= box_size)
        xs_ref = np.where(bad, xs, xs_ref)
        ys_ref = np.where(bad, ys, ys_ref)
        return xs_ref, ys_ref
    except Exception as e:
        print(f"[CENTROID REFINE] {e}")
        return xs, ys


def compute_adaptive_apertures(xs, ys, default_r=DEFAULT_APERTURE_R,
                                min_r=MIN_APERTURE_R, safety_buffer=APERTURE_SAFETY_BUFFER):
    """Crowding-aware aperture radius per source: shrinks a source's
    aperture only if a neighbor is close enough that default_r apertures
    would physically overlap and contaminate each other's flux."""
    n = len(xs)
    if n == 0:
        return np.array([])
    if n == 1:
        return np.array([default_r])

    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    min_sep = np.full(n, np.inf)
    for i in range(n):
        d = np.hypot(xs - xs[i], ys - ys[i])
        d[i] = np.inf
        min_sep[i] = d.min()

    aper_radii = np.minimum(default_r, min_sep / 2 - safety_buffer)
    return np.clip(aper_radii, min_r, default_r)


def local_background_photometry(data, xs, ys, aper_radii, annulus_width=ANNULUS_WIDTH):
    """Per-source aperture photometry with a LOCAL background estimated
    from a sigma-clipped annulus around each source, excluding any pixels
    that fall inside a NEIGHBORING source's own aperture. Bounding-box
    slices are used throughout (not full-image allocations per source) so
    this stays cheap even for plates with hundreds of sources. Returns
    (final_flux, raw_aperture_sum, bkg_std, apertures)."""
    n = len(xs)
    positions = list(zip(xs, ys))
    apertures = [CircularAperture(positions[i], r=float(aper_radii[i])) for i in range(n)]
    annuli = [
        CircularAnnulus(positions[i], r_in=float(aper_radii[i]), r_out=float(aper_radii[i]) + annulus_width)
        for i in range(n)
    ]

    combined_star_mask = np.zeros(data.shape, dtype=bool)
    for ap in apertures:
        m = ap.to_mask(method="center")
        sl_large, sl_small = m.get_overlap_slices(data.shape)
        if sl_large is None:
            continue
        combined_star_mask[sl_large] |= (m.data[sl_small] > 0)

    raw_sum = np.zeros(n)
    final_flux = np.zeros(n)
    bkg_std = np.full(n, np.nan)

    for i in range(n):
        t = aperture_photometry(data, apertures[i])
        raw_sum[i] = float(t["aperture_sum"][0])

        ann_mask_obj = annuli[i].to_mask(method="center")
        sl_large, sl_small = ann_mask_obj.get_overlap_slices(data.shape)

        bkg_median = 0.0
        if sl_large is not None:
            local_ring = ann_mask_obj.data[sl_small] > 0
            local_star = combined_star_mask[sl_large]
            local_data = data[sl_large]
            clean = local_ring & (~local_star)
            ann_pixels = local_data[clean]
            if ann_pixels.size >= 10:
                _, bkg_median, bkg_sigma = sigma_clipped_stats(ann_pixels)
                bkg_std[i] = bkg_sigma

        final_flux[i] = raw_sum[i] - bkg_median * apertures[i].area

    return final_flux, raw_sum, bkg_std, apertures


def detect_source_saturation(data, xs, ys, apertures,
                              sat_fraction=SATURATION_FRACTION, sat_min_pixels=SATURATION_MIN_PIXELS):
    """Flags a source as saturated if too many pixels inside ITS OWN
    aperture sit within sat_fraction of that source's own peak value --
    i.e. a flat-topped profile rather than a normal PSF falloff."""
    n = len(xs)
    is_saturated = np.zeros(n, dtype=bool)
    max_vals = np.full(n, np.nan)

    for i in range(n):
        mask = apertures[i].to_mask(method="center")
        cut = mask.multiply(data)
        if cut is None:
            continue
        star_pixels = cut[mask.data > 0]
        if star_pixels.size == 0:
            continue
        max_val = float(star_pixels.max())
        max_vals[i] = max_val
        near_max_count = int(np.sum(star_pixels >= max_val * sat_fraction))
        is_saturated[i] = near_max_count > sat_min_pixels

    return is_saturated, max_vals


def compute_wings_fwhm(data, xs, ys, is_saturated, max_vals, sat_fraction=SATURATION_FRACTION,
                        edge_radii=FWHM_EDGE_RADII):
    """Gaussian FWHM fit to the unsaturated WINGS of each saturated
    source's radial profile (core pixels masked out). Only computed for
    sources flagged saturated -- an unsaturated source's peak pixel value
    is already informative enough without this."""
    n = len(xs)
    fwhm = np.full(n, np.nan)

    for i in range(n):
        if not is_saturated[i] or not np.isfinite(max_vals[i]):
            continue
        try:
            sat_thresh = max_vals[i] * sat_fraction
            sat_mask = data >= sat_thresh
            rp = RadialProfile(data, (float(xs[i]), float(ys[i])), edge_radii, mask=sat_mask)
            rp.gaussian_fit
            fwhm[i] = float(rp.gaussian_fwhm)
        except Exception as e:
            print(f"[WINGS FWHM] source at ({xs[i]:.1f},{ys[i]:.1f}): {e}")

    return fwhm


def compute_plate_limit(bkg_std, calibration, n_sigma=PLATE_LIMIT_NSIGMA, reference_r=DEFAULT_APERTURE_R):
    """Self-derived N-sigma point-source detection limit, converted to a
    real B magnitude via this plate's own instrumental-to-catalog fit.
    Purely a diagnostic field -- does NOT feed into
    classify_plate_quality(), which continues to rely on DASCH's own
    lim_mag_apass/lim_mag_atlas. Returns None if there's no usable
    background noise or calibration to convert with."""
    valid_std = bkg_std[np.isfinite(bkg_std)]
    if valid_std.size == 0 or calibration is None:
        return None
    mean_bkg_std = float(np.mean(valid_std))
    reference_area = np.pi * reference_r ** 2
    threshold_flux = n_sigma * mean_bkg_std * np.sqrt(reference_area)
    if threshold_flux <= 0:
        return None
    threshold_inst_mag = -2.5 * np.log10(threshold_flux)
    slope = calibration["slope"]
    if slope == 0:
        return None
    return (threshold_inst_mag - calibration["intercept"]) / slope


def compute_differential_photometry(final_flux, catalog_b, is_saturated):
    """Two-star differential photometry: anchors to the brightest
    unsaturated, APASS-matched source and transfers its real catalog B
    magnitude to every other source using only the flux ratio between
    that pair -- independent of the line fit above, so it doubles as a
    sanity check (compare derived_b against catalog_b wherever both
    exist). Returns (derived_b array, reference index or None)."""
    n = len(final_flux)
    derived_b = np.full(n, np.nan)

    candidates = [
        i for i in range(n)
        if np.isfinite(catalog_b[i]) and not is_saturated[i] and np.isfinite(final_flux[i]) and final_flux[i] > 0
    ]
    if not candidates:
        return derived_b, None

    ref = candidates[int(np.argmax(final_flux[candidates]))]
    ref_flux = final_flux[ref]

    for i in range(n):
        if i == ref:
            derived_b[i] = catalog_b[ref]
            continue
        if not (np.isfinite(final_flux[i]) and final_flux[i] > 0):
            continue
        delta_mag = -2.5 * np.log10(final_flux[i] / ref_flux)
        derived_b[i] = catalog_b[ref] + delta_mag

    return derived_b, ref


def full_scan(plates=None):
    """
    Expensive pass: robust per-source photometry (centroid refinement,
    crowding-aware adaptive apertures, local background via CircularAnnulus,
    saturation detection + wings-only FWHM, self-derived plate limit,
    differential-photometry cross-check), Gaia/SIMBAD/APASS crossmatch, a
    calibration fit (now excluding saturated sources), and quality
    refinement. Only meant to run AFTER prescan() has populated plate_db
    and a paradigm plate has been labeled -- see 'Apply Labels', which
    gates this behind paradigm_established(). SIMBAD is queried once per
    plate pointing (get_plate_catalog_simbad), not once per source.
    """
    global _full_scan_has_run

    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    scan_progress.max = total
    scan_progress.value = 0
    start_time = time.monotonic()
    _tick_state = {"last_render": 0.0}

    def _tick(n_done):
        scan_progress.value = n_done
        now = time.monotonic()
        if now - _tick_state["last_render"] > 0.15 or n_done == total:
            render_scan_progress(n_done, total, start_time, status_word="Full scanning", color="#2196F3")
            _tick_state["last_render"] = now

    for i, f in enumerate(target_plates):

        cached = plate_db.get(str(f))
        if cached is None or cached.get("render") is None:
            _tick(i + 1)
            continue

        r = cached["render"]
        photometry_current = r.get("photometry_version", -1) == PHOTOMETRY_VERSION

        if r.get("phase") == "full" and photometry_current and r.get("paradigm_version", -1) == paradigm_version:
            _tick(i + 1)
            continue

        if r.get("phase") == "full" and photometry_current and r.get("paradigm_version", -1) != paradigm_version:
            # Cheap path : only re-resolve names against the (possibly
            # updated) paradigm_db, so no need to redo Gaia/SIMBAD/APASS
            # or any of the photometry below.
            hdr = astrofits.getheader(f)
            wcs = WCS(hdr)
            ra, dec = wcs.pixel_to_world_values(r["sources"][r["x_col"]], r["sources"][r["y_col"]])
            ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)

            paradigm_names = match_against_paradigm(ra, dec)
            r["paradigm_names"] = paradigm_names
            r["resolved_names"] = [
                p if p != "Unknown" else old
                for p, old in zip(paradigm_names, r["resolved_names"])
            ]
            r["resolved_names"] = [resolve_gaia_label(name) for name in r["resolved_names"]]
            r["paradigm_version"] = paradigm_version
            _tick(i + 1)
            continue

        # phase == "prescan", OR phase == "full" but photometry_version is
        # stale (needs the new centroid/aperture/background/saturation
        # pipeline re-run even though it was already fully scanned once) :
        # needs the full expensive treatment.
        scan_status.value = f.name

        try:
            sources = r["sources"]
            x_col, y_col = r["x_col"], r["y_col"]
            n = len(sources)

            hdr = astrofits.getheader(f)
            wcs = WCS(hdr)

            data = astrofits.getdata(f).astype(float)
            data = np.nan_to_num(data, nan=np.nanmedian(data))

            # Centroid refinement -- overwrites each source's stored pixel
            # position in place, so everything downstream (RA/Dec, catalog
            # crossmatch, photometry, plotting) benefits from the fix.
            xs_init = np.array(sources[x_col], dtype=float)
            ys_init = np.array(sources[y_col], dtype=float)
            xs_ref, ys_ref = refine_source_centroids(data, xs_init, ys_init)
            sources[x_col] = xs_ref
            sources[y_col] = ys_ref

            # Crowding-aware adaptive aperture radius per source.
            aper_radii = compute_adaptive_apertures(xs_ref, ys_ref)

            # Local per-source background via CircularAnnulus, excluding
            # any neighboring source's own aperture from that ring --
            # replaces the old single-global-median background subtraction.
            final_flux, raw_flux, bkg_std, apertures_list = local_background_photometry(
                data, xs_ref, ys_ref, aper_radii
            )

            # Saturation detection + wings-only FWHM for saturated sources.
            is_saturated, max_vals = detect_source_saturation(data, xs_ref, ys_ref, apertures_list)
            fwhm_values = compute_wings_fwhm(data, xs_ref, ys_ref, is_saturated, max_vals)

            flux_for_mag = np.where(final_flux > 0, final_flux, np.nan)
            inst_mag = -2.5 * np.log10(flux_for_mag)

            ra, dec = wcs.pixel_to_world_values(xs_ref, ys_ref)
            ra = np.array(ra, dtype=float)
            dec = np.array(dec, dtype=float)

            gaia_catalog = get_plate_catalog_gaia(wcs, data.shape)
            apass_catalog = get_plate_catalog_apass(wcs, data.shape)
            simbad_catalog = get_plate_catalog_simbad(wcs, data.shape)

            src_coords = SkyCoord(ra * u.deg, dec * u.deg)
            catalog_b = np.full(len(src_coords), np.nan)

            if apass_catalog is not None and len(apass_catalog) > 0:
                apass_coords = SkyCoord(apass_catalog["ra"], apass_catalog["dec"], unit="deg")
                idx_src, idx_cat, sep2d, _ = search_around_sky(
                    src_coords, apass_coords, seplimit=5 * u.arcsec
                )
                for s, c in zip(idx_src, idx_cat):
                    catalog_b[s] = apass_catalog["b_mag"][c]

            gaia_names = match_detected_sources_gaia(ra, dec, gaia_catalog)
            simbad_names = match_detected_sources_simbad(ra, dec, simbad_catalog)

            update_name_cache(gaia_names, simbad_names)
            resolved_names = resolve_names(gaia_names, simbad_names)

            paradigm_names = match_against_paradigm(ra, dec)
            resolved_names = [
                p if p != "Unknown" else r2
                for p, r2 in zip(paradigm_names, resolved_names)
            ]
            resolved_names = [resolve_gaia_label(name) for name in resolved_names]

            matched_mask = np.array([g != "Unknown" for g in gaia_names]) | np.isfinite(catalog_b)
            n_matched = int(np.sum(matched_mask))

            lim_mag_apass, lim_mag_atlas = get_plate_limits(f)
            quality = classify_plate_quality(
                r["errors"], n, n_matched, lim_mag_apass, lim_mag_atlas
            )

            r.update({
                "inst_mag":           inst_mag,
                "catalog_b":          catalog_b,
                "aper_radii":         aper_radii,
                "bkg_std":            bkg_std,
                "is_saturated":       is_saturated,
                "fwhm":               fwhm_values,
                "gaia_names":         gaia_names,
                "simbad_names":       simbad_names,
                "resolved_names":     resolved_names,
                "paradigm_names":     paradigm_names,
                "paradigm_version":   paradigm_version,
                "photometry_version": PHOTOMETRY_VERSION,
                "phase":              "full",
            })

            # Linearity fit -- saturated sources excluded (unreliable
            # flux), same idea as before but now saturation-aware.
            x_all = np.array(inst_mag)
            y_all = np.array(catalog_b)
            calib_mask = np.isfinite(x_all) & np.isfinite(y_all) & (~is_saturated)
            r["calib_mask"] = calib_mask
            x = x_all[calib_mask]
            y = y_all[calib_mask]
            if len(x) >= 20:
                slope, intercept = np.polyfit(y, x, 1)
                r["calibration"] = {"slope": slope, "intercept": intercept, "n_used": len(x)}
            else:
                r["calibration"] = None

            # Self-derived 5-sigma plate limit -- purely a diagnostic
            # field; classify_plate_quality() above is untouched and still
            # relies only on DASCH's own lim_mag_apass/lim_mag_atlas.
            r["self_plate_limit_b"] = compute_plate_limit(bkg_std, r["calibration"])

            # Two-star differential-photometry cross-check against APASS.
            derived_b, diff_ref_idx = compute_differential_photometry(final_flux, catalog_b, is_saturated)
            r["derived_b"] = derived_b
            r["diff_ref_idx"] = diff_ref_idx

            plate_db[str(f)]["quality"] = quality
            plate_db[str(f)]["render"] = r

        except Exception as e:
            print(f"[FULL SCAN ERROR] {f.name}: {e}")

        if (i + 1) % 10 == 0:
            with open(PLATE_DB_FILE, "wb") as fp:
                pickle.dump(plate_db, fp)
            print(f"Full-scan autosaved {i + 1} plates")

        enrich_previous_plates(plate_db, str(f))
        _tick(i + 1)

    render_scan_progress(total, total, start_time, status_word="Full scan complete", color="#2196F3")
    scan_status.value = "Full scan complete"
    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)
    print(f"Full-scanned plates saved ({len(plate_db)} total in plate_db)")
    _full_scan_has_run = True
    refresh_filter_options()
    rebuild_dropdown()

# Shared filtering infrastructure — the paradigm plate picker and the main
# viewer's Plate dropdown each get their OWN independent Filter + Search
# widgets, so choosing a category for one doesn't restrict the other.
category_filter = widgets.Dropdown(
    options=["all", "ideal", "good_target", "good_no_target", "defective_target", "defective_no_target"],
    value="all",
    description="Filter:"
)
search_box = widgets.Text(
    description="Search:",
    placeholder="filter by filename, e.g. 0012"
)

main_category_filter = widgets.Dropdown(
    options=["all", "ideal", "good_target", "good_no_target", "defective_target", "defective_no_target"],
    value="all",
    description="Filter:"
)
main_search_box = widgets.Text(
    description="Search:",
    placeholder="filter by filename, e.g. 0012"
)

dropdown = widgets.Dropdown(description="Plate:")
dropdown_status = widgets.Label(value="")
paradigm_dropdown_status = widgets.Label(value="")

MAX_DROPDOWN_OPTIONS = 500

QUALITY_RANK = {"good_match": 0, "fair": 1, "too_many_errors": 2, "defective": 3}

CATEGORY_DISPLAY = {
    "ideal": "Ideal Image",
    "good_target": "Good Image, Target",
    "good_no_target": "Good Image, No Target",
    "defective_target": "Defective Image, Target",
    "defective_no_target": "Defective Image, No Target",
}

def categorize_plate(meta):
    """Map a plate_db entry to exactly one of the five Filter categories.
    A human 'approved' verdict on an algorithmically 'fair' plate elevates
    it to the same tier as 'good_match' for categorization purposes only
    -- meta['quality'] itself is left untouched."""
    quality = meta.get("quality", "fair")
    if meta.get("human_verdict") == "approved" and quality == "fair":
        quality = "good_match"
    target = bool(meta.get("target", False))
    is_good = quality in ("good_match", "fair")

    if quality == "good_match" and target:
        return "ideal"
    if target:
        return "good_target" if is_good else "defective_target"
    else:
        return "good_no_target" if is_good else "defective_no_target"

def filter_plates(cat_value, query, exclude_rejected=False):
    """Shared filtering logic used by both the paradigm picker and the
    main viewer dropdown. exclude_rejected=True additionally drops any
    plate a human reviewer marked "rejected" -- used by the paradigm
    picker only, so rejected plates still remain browsable in the main
    viewer once it's enabled."""

    query = query.strip().lower()
    matches = []

    for f_str, meta in plate_db.items():
        f = Path(f_str)

        if exclude_rejected and meta.get("human_verdict") == "rejected":
            continue

        cat = categorize_plate(meta)
        if cat_value != "all" and cat != cat_value:
            continue
        if query and query not in f.name.lower():
            continue

        label = f"{f.name} | {CATEGORY_DISPLAY[cat]} | {meta['n']} sources | {meta['date']}"
        q_rank = QUALITY_RANK.get(meta.get("quality", "fair"), 1)
        sort_key = (q_rank, -meta["n"])
        matches.append((label, f, sort_key))

    for f in cutouts:
        if str(f) not in plate_db:
            if query and query not in f.name.lower():
                continue
            if cat_value != "all":
                continue
            matches.append((f"{f.name} | not yet scanned", f, (99, 0)))

    matches.sort(key=lambda m: m[2])
    return [(label, f) for label, f, _ in matches]

def rebuild_paradigm_dropdown(*args):
    matches = filter_plates(category_filter.value, search_box.value, exclude_rejected=True)
    total_matches = len(matches)
    options = matches[:MAX_DROPDOWN_OPTIONS]
    paradigm_plate_dropdown.options = options

    if total_matches > MAX_DROPDOWN_OPTIONS:
        paradigm_dropdown_status.value = (
            f"Showing first {MAX_DROPDOWN_OPTIONS} of {total_matches} matches — "
            f"narrow with Search or the filter above to see the rest."
        )
    else:
        paradigm_dropdown_status.value = f"{total_matches} matches"

def rebuild_main_dropdown(*args):
    if not _full_scan_has_run:
        dropdown.options = []
        dropdown_status.value = (
            "Label a paradigm plate above, then click 'Apply Labels' to "
            "run the full scan and enable this viewer."
        )
        return

    matches = filter_plates(main_category_filter.value, main_search_box.value, exclude_rejected=False)
    total_matches = len(matches)
    options = matches[:MAX_DROPDOWN_OPTIONS]
    dropdown.options = options

    if total_matches > MAX_DROPDOWN_OPTIONS:
        dropdown_status.value = (
            f"Showing first {MAX_DROPDOWN_OPTIONS} of {total_matches} matches — "
            f"narrow with Search or the filter above to see the rest."
        )
    else:
        dropdown_status.value = f"{total_matches} matches"

def rebuild_dropdown(*args):
    rebuild_paradigm_dropdown()
    rebuild_main_dropdown()

def refresh_filter_options():
    """Recompute per-category plate counts from plate_db and bake them
    into BOTH Filter dropdowns' labels."""

    counts = {
        "ideal": 0,
        "good_target": 0,
        "good_no_target": 0,
        "defective_target": 0,
        "defective_no_target": 0,
    }
    total = len(plate_db)

    for meta in plate_db.values():
        cat = categorize_plate(meta)
        counts[cat] = counts.get(cat, 0) + 1

    new_options = [
        (f"all ({total})", "all"),
        (f"{CATEGORY_DISPLAY['ideal']} ({counts['ideal']})", "ideal"),
        (f"{CATEGORY_DISPLAY['good_target']} ({counts['good_target']})", "good_target"),
        (f"{CATEGORY_DISPLAY['good_no_target']} ({counts['good_no_target']})", "good_no_target"),
        (f"{CATEGORY_DISPLAY['defective_target']} ({counts['defective_target']})", "defective_target"),
        (f"{CATEGORY_DISPLAY['defective_no_target']} ({counts['defective_no_target']})", "defective_no_target"),
    ]

    for dd in (category_filter, main_category_filter):
        cur_val = dd.value
        dd.options = new_options
        dd.value = cur_val

category_filter.observe(rebuild_paradigm_dropdown, names="value")
search_box.observe(rebuild_paradigm_dropdown, names="value")
main_category_filter.observe(rebuild_main_dropdown, names="value")
main_search_box.observe(rebuild_main_dropdown, names="value")

refresh_filter_options()


# Review uncertain plates -- surfaced BEFORE paradigm plate selection, so
# the shortlist categories above are as accurate as possible by the time
# a paradigm plate is picked. Instead of reviewing every single "fair"
# plate individually, unreviewed "fair" plates are clustered by similarity
# (defect-type breakdown, dead-zone/saturation extent, how far off the
# limiting magnitude is from the dataset median, source density) and only
# ONE representative per cluster is surfaced. Approving/Rejecting that
# representative applies the same verdict to every other plate judged
# similar to it -- so reviewing a handful of representatives resolves the
# whole backlog instead of clicking through a thousand individual plates.

REVIEW_CLUSTERS = 12  # how many representative plates to surface. Lower
                       # this for fewer/broader groups, raise it for more/
                       # finer-grained groups.

_fair_cluster_of = {}       # { file_str: cluster_id }
_fair_cluster_members = {}  # { cluster_id: [file_str, ...] }

def _fair_plate_features(f_str, meta):
    """8-feature vector describing WHY a plate landed in 'fair', used to
    group similar borderline plates together."""
    r = meta.get("render") or {}
    errs = r.get("errors", {})
    lim_apass, lim_atlas = get_plate_limits(Path(f_str))
    lim = lim_apass if lim_apass is not None else lim_atlas
    ref = _lim_mag_median_apass if lim_apass is not None else _lim_mag_median_atlas
    lim_z = float(lim - ref) if (lim is not None and ref is not None) else 0.0
    return np.array([
        len(errs.get("scratches", [])),
        len(errs.get("trailing", [])),
        len(errs.get("saturation", [])),
        len(errs.get("dust", [])),
        errs.get("dead_zone_fraction", 0.0) * 10.0,
        errs.get("saturation_area_fraction", 0.0) * 10.0,
        lim_z,
        float(np.log1p(meta.get("n", 0))),
    ], dtype=float)

def _cluster_fair_plates(n_clusters=REVIEW_CLUSTERS):
    """Groups every unreviewed 'fair' plate by similarity, picks ONE
    representative per cluster (closest to that cluster's centroid), and
    returns just those representatives. Also populates _fair_cluster_of /
    _fair_cluster_members so a decision on a representative can be applied
    to its whole cluster."""
    global _fair_cluster_of, _fair_cluster_members

    _fair_cluster_of = {}
    _fair_cluster_members = {}

    fair_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if meta.get("quality") == "fair"
        and meta.get("human_verdict") is None
        and meta.get("render") is not None
    ]

    if not fair_items:
        return []

    if len(fair_items) <= 1:
        f_str = fair_items[0][0]
        _fair_cluster_of[f_str] = 0
        _fair_cluster_members[0] = [f_str]
        return [f_str]

    feats = np.array([_fair_plate_features(f_str, meta) for f_str, meta in fair_items])

    # Z-score each feature column, with a safe fallback for zero-variance
    # columns (e.g. dead_zone_fraction is usually ~0 across the whole
    # "fair" bucket, since anything with a real dead zone was already
    # hard-failed to "defective" upstream).
    means = feats.mean(axis=0)
    stds = feats.std(axis=0)
    stds[stds < 1e-9] = 1.0
    feats_z = (feats - means) / stds
    feats_z = np.nan_to_num(feats_z, nan=0.0, posinf=0.0, neginf=0.0)

    k = max(1, min(n_clusters, len(fair_items)))

    try:
        from scipy.cluster.vq import kmeans2
        centroids, labels = kmeans2(feats_z, k, minit="++", seed=0)
    except Exception as e:
        print(f"[REVIEW CLUSTERING] {e} -- falling back to a simple round-robin grouping")
        labels = np.arange(len(fair_items)) % k

    representatives = []
    for cluster_id in range(k):
        member_idx = np.where(labels == cluster_id)[0]
        if len(member_idx) == 0:
            continue
        member_files = [fair_items[i][0] for i in member_idx]
        for mf in member_files:
            _fair_cluster_of[mf] = cluster_id
        _fair_cluster_members[cluster_id] = member_files

        centroid = feats_z[member_idx].mean(axis=0)
        dists = np.linalg.norm(feats_z[member_idx] - centroid, axis=1)
        rep_idx = member_idx[int(np.argmin(dists))]
        representatives.append(fair_items[rep_idx][0])

    return representatives

review_plate_dropdown = widgets.Dropdown(description="Needs review:")
review_status = widgets.Label(value="")
review_output = widgets.Output()
approve_btn = widgets.Button(description="Approve (treat as good)", button_style="success")
reject_btn = widgets.Button(description="Reject (exclude from shortlist)", button_style="danger")
review_instructions = widgets.HTML(
    "<small>These are representative 'fair'-quality plates -- one per "
    "cluster of similar plates, not every single one. Each label shows "
    "how many plates it represents. <b>Approve</b> treats this "
    "representative AND every plate clustered with it as good "
    "(equivalent to 'good_match'); <b>Reject</b> excludes this "
    "representative and its whole cluster from the paradigm-plate "
    "shortlist below (they stay browsable in the main viewer once "
    "that's enabled). Reviewing is optional -- unreviewed plates are "
    "just left out of the 'Ideal Image'/'Good Image' categories until "
    "you decide.</small>"
)

def _pending_review_plates():
    representatives = _cluster_fair_plates()
    items = []
    for f_str in representatives:
        meta = plate_db.get(f_str)
        if meta is None:
            continue
        f = Path(f_str)
        r = meta["render"]
        errs = r.get("errors", {})
        cluster_id = _fair_cluster_of.get(f_str)
        n_members = len(_fair_cluster_members.get(cluster_id, [f_str]))
        label = (
            f"{f.name} | represents {n_members} similar plate(s) | "
            f"{meta['n']} sources | "
            f"dead_zone {errs.get('dead_zone_fraction', 0.0) * 100:.0f}% | "
            f"defects: scr{len(errs.get('scratches', []))} "
            f"trail{len(errs.get('trailing', []))} "
            f"sat{len(errs.get('saturation', []))} "
            f"dust{len(errs.get('dust', []))}"
        )
        items.append((n_members, label, f))
    items.sort(key=lambda t: -t[0])
    return [(label, f) for _, label, f in items]

def _refresh_review_queue():
    pending = _pending_review_plates()
    review_plate_dropdown.options = pending
    review_status.value = f"{len(pending)} representative plate(s) awaiting review."

def _open_review_plate(change):
    with review_output:
        clear_output(wait=True)
        fits_path = review_plate_dropdown.value
        if fits_path is None:
            print("No plates currently need review.")
            return

        meta = plate_db.get(str(fits_path))
        if meta is None or meta.get("render") is None:
            print("No data available for this plate.")
            return
        r = meta["render"]
        data = astrofits.getdata(fits_path)

        cluster_id = _fair_cluster_of.get(str(fits_path))
        n_members = len(_fair_cluster_members.get(cluster_id, [str(fits_path)]))

        # Rendered as a STATIC PNG, not an interactive %matplotlib widget
        # figure. This panel doesn't need clicking (Approve/Reject are
        # separate buttons), and having a second concurrently-open
        # interactive figure was the actual cause of a real bug: ipympl
        # only reliably routes mouse events to ONE interactive canvas at a
        # time, so opening a plate here was silently stealing click focus
        # away from the Paradigm labeling plot below.
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(data, origin="lower", cmap="gray")
        ax.scatter(
            r["sources"][r["x_col"]], r["sources"][r["y_col"]],
            s=40, facecolors="none", edgecolors="red"
        )
        if r.get("found"):
            ax.scatter(
                r["sources"][r["x_col"]][r["target_idx"]],
                r["sources"][r["y_col"]][r["target_idx"]],
                s=120, edgecolors="cyan", facecolors="none"
            )
        elif r.get("target_fallback"):
            fb = r["target_fallback"]
            ax.scatter([fb["x"]], [fb["y"]], s=140, edgecolors="cyan", facecolors="none", linewidths=2)
        annotate_errors(ax, r.get("errors", {}))
        ax.set_title(f"{fits_path.name}  (represents {n_members} similar plate(s))")

        buf = BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        display(widgets.Image(value=buf.read(), format="png"))

        lim_mag_apass, lim_mag_atlas = get_plate_limits(fits_path)
        print(f"n_sources={meta['n']}   target={meta['target']}   tag={meta['tag']}")
        print(f"lim_mag_apass={lim_mag_apass}   lim_mag_atlas={lim_mag_atlas}")
        print(f"dead_zone_fraction={r.get('errors', {}).get('dead_zone_fraction', 0.0) * 100:.1f}%")
        print(f"saturation_area_fraction={r.get('errors', {}).get('saturation_area_fraction', 0.0) * 100:.1f}%")
        print(f"Deciding this plate will apply to all {n_members} plate(s) in its cluster.")

def _approve_plate(_):
    fits_path = review_plate_dropdown.value
    if fits_path is None:
        return
    cluster_id = _fair_cluster_of.get(str(fits_path))
    members = _fair_cluster_members.get(cluster_id, [str(fits_path)])
    for f_str in members:
        if f_str in plate_db:
            plate_db[f_str]["human_verdict"] = "approved"
    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)
    with review_output:
        clear_output(wait=True)
        print(f"Approved {len(members)} plate(s) (this representative + everything clustered with it).")
    _refresh_review_queue()
    refresh_filter_options()
    rebuild_paradigm_dropdown()

def _reject_plate(_):
    fits_path = review_plate_dropdown.value
    if fits_path is None:
        return
    cluster_id = _fair_cluster_of.get(str(fits_path))
    members = _fair_cluster_members.get(cluster_id, [str(fits_path)])
    for f_str in members:
        if f_str in plate_db:
            plate_db[f_str]["human_verdict"] = "rejected"
    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)
    with review_output:
        clear_output(wait=True)
        print(f"Rejected {len(members)} plate(s) (this representative + everything clustered with it).")
    _refresh_review_queue()
    refresh_filter_options()
    rebuild_paradigm_dropdown()

review_plate_dropdown.observe(_open_review_plate, names="value")
approve_btn.on_click(_approve_plate)
reject_btn.on_click(_reject_plate)

review_panel = widgets.VBox([
    widgets.HTML("<b>Review uncertain plates</b>"),
    review_instructions,
    review_plate_dropdown,
    review_status,
    widgets.HBox([approve_btn, reject_btn]),
    review_output,
])

display(review_panel)
_refresh_review_queue()

# Paradigm plate labeling UI — supports manual add/remove of detections
paradigm_plate_dropdown = widgets.Dropdown(description="Paradigm plate:")
auto_load_btn = widgets.Button(description="Auto-Load Stars", button_style="info")
clear_plate_btn = widgets.Button(description="Clear Plate", button_style="danger")
copy_name_btn = widgets.Button(description="Copy Name → Search", button_style="")
apply_labels_btn = widgets.Button(description="Apply Labels", button_style="success")
paradigm_status = widgets.Label(value="")
paradigm_instructions = widgets.HTML(
    "<small>Use the Filter above (and the Review panel above that, for "
    "any 'fair'-quality plates) to narrow down to a strong candidate, "
    "then select it in 'Paradigm plate:' below -- it opens automatically "
    "with no markers. 'Auto-Load Stars' runs the detection algorithm and "
    "adds the found sources as red markers (skipping any too close to a "
    "marker you already placed). Left-click a red marker to label it. "
    "Left-click empty space to <b>add</b> a manual (orange) detection "
    "there -- the label popup appears immediately, and the SIMBAD/Gaia "
    "name suggestion fills in a moment later in the background, so you "
    "never have to wait on the network just to keep clicking. Right-click "
    "a marker to <b>remove</b> it from this plate's source list. "
    "'Clear Plate' removes every marker and every saved label for this "
    "plate. 'Copy Name → Search' puts this plate's filename into the "
    "Search box above. Label as many objects as you like, then click "
    "'Apply Labels' <b>once</b> -- this runs the full Gaia/SIMBAD/APASS "
    "scan across every plate and is what enables the main viewer at the "
    "bottom (disabled until then).</small>"
)
paradigm_plot_output = widgets.Output()
paradigm_output = widgets.Output()
paradigm_table_output = widgets.Output()

_paradigm_state = {
    "fig": None, "ax": None, "scatter": None, "fits_path": None,
    "xs": [], "ys": [], "manual": [], "wcs": None, "data": None,
}

def _nearest_source(x_click, y_click, max_px=PARADIGM_CLICK_PX):
    xs, ys = _paradigm_state["xs"], _paradigm_state["ys"]
    if not xs:
        return None, None
    xs_arr, ys_arr = np.array(xs), np.array(ys)
    d = np.hypot(xs_arr - x_click, ys_arr - y_click)
    j = int(np.argmin(d))
    return (j, d[j]) if d[j] <= max_px else (None, None)

def _redraw_sources():
    ax = _paradigm_state["ax"]
    if ax is None:
        return
    if _paradigm_state["scatter"] is not None:
        _paradigm_state["scatter"].remove()

    xs, ys, manual = _paradigm_state["xs"], _paradigm_state["ys"], _paradigm_state["manual"]
    colors = ["orange" if m else "red" for m in manual]
    _paradigm_state["scatter"] = ax.scatter(
        xs, ys, s=50, facecolors="none", edgecolors=colors, linewidths=1.5
    )
    ax.figure.canvas.draw_idle()

def _label_point(x, y, ra, dec):
    """Opens the label popup IMMEDIATELY (no network wait), then runs the
    SIMBAD/Gaia name suggestion in a background thread and fills it in
    once it resolves."""

    label_type = widgets.ToggleButtons(
        options=["Object ID", "GAIA ID"],
        value="Object ID",
        description="Type:"
    )
    label_box = widgets.Text(
        value="",
        description="Label:",
        placeholder="looking up suggestion... (or just start typing)"
    )
    confirm_btn = widgets.Button(description="Confirm", button_style="success")
    skip_btn = widgets.Button(description="Skip", button_style="")

    def _confirm(_):
        raw = label_box.value.strip()

        if label_type.value == "GAIA ID":
            digits = raw.replace("Gaia", "").strip()
            if digits.isdigit():
                final_label = f"Gaia {digits}"
            else:
                with paradigm_output:
                    clear_output(wait=True)
                    print(f"'{raw}' isn't a numeric Gaia source_id — enter digits only, e.g. 123456789012345.")
                return
        else:
            final_label = raw

        paradigm_db.setdefault(str(_paradigm_state["fits_path"]), [])
        paradigm_db[str(_paradigm_state["fits_path"])].append({
            "ra": ra, "dec": dec, "label": final_label, "source": "manual/suggested"
        })
        save_paradigm_db()
        bump_paradigm_version()

        with paradigm_output:
            clear_output(wait=True)
            print(f"Saved: {final_label}  (RA={ra:.5f}, Dec={dec:.5f})")
            print("Label more objects on this plate, then click 'Apply Labels' when done.")
        _refresh_paradigm_table()

    def _skip(_):
        with paradigm_output:
            clear_output(wait=True)

    confirm_btn.on_click(_confirm)
    skip_btn.on_click(_skip)

    with paradigm_output:
        clear_output(wait=True)
        print(f"Point at pixel ({x:.1f}, {y:.1f})  ->  RA={ra:.5f}, Dec={dec:.5f}")
        print("Looking up a name suggestion in the background...")
        display(widgets.HBox([label_type, label_box, confirm_btn, skip_btn]))

    def _do_lookup():
        try:
            suggestion, src = suggest_name_at(ra, dec)
        except Exception as e:
            suggestion, src = "Unknown", f"error: {e}"

        # Don't stomp on anything the user already typed while waiting.
        if not label_box.value.strip():
            suggested_mode = "GAIA ID" if suggestion.startswith("Gaia ") else "Object ID"
            label_type.value = suggested_mode
            label_box.value = suggestion.replace("Gaia ", "") if suggested_mode == "GAIA ID" else suggestion

        with paradigm_output:
            print(f"Auto-suggested: '{suggestion}' (source: {src})")

    Thread(target=_do_lookup, daemon=True).start()

def _on_plate_click(event):
    if event.inaxes != _paradigm_state["ax"] or event.xdata is None:
        return

    wcs = _paradigm_state["wcs"]

    if event.button == 3:
        j, dist = _nearest_source(event.xdata, event.ydata)
        if j is None:
            with paradigm_output:
                clear_output(wait=True)
                print("No marker close enough to remove.")
            return
        removed_x, removed_y = _paradigm_state["xs"].pop(j), _paradigm_state["ys"].pop(j)
        _paradigm_state["manual"].pop(j)
        ra, dec = wcs.pixel_to_world_values(removed_x, removed_y)
        ra, dec = float(np.array(ra)), float(np.array(dec))

        entries = paradigm_db.get(str(_paradigm_state["fits_path"]), [])
        entries = [e for e in entries if not (abs(e["ra"] - ra) < 1e-7 and abs(e["dec"] - dec) < 1e-7)]
        paradigm_db[str(_paradigm_state["fits_path"])] = entries
        save_paradigm_db()
        bump_paradigm_version()

        _redraw_sources()
        with paradigm_output:
            clear_output(wait=True)
            print(f"Removed detection at pixel ({removed_x:.1f}, {removed_y:.1f}).")
        _refresh_paradigm_table()
        return

    j, dist = _nearest_source(event.xdata, event.ydata)

    if j is not None:
        x, y = _paradigm_state["xs"][j], _paradigm_state["ys"][j]
    else:
        x, y = float(event.xdata), float(event.ydata)
        _paradigm_state["xs"].append(x)
        _paradigm_state["ys"].append(y)
        _paradigm_state["manual"].append(True)
        _redraw_sources()

    ra, dec = wcs.pixel_to_world_values(x, y)
    ra, dec = float(np.array(ra)), float(np.array(dec))
    _label_point(x, y, ra, dec)

def _refresh_paradigm_table():
    with paradigm_table_output:
        clear_output(wait=True)
        entries = paradigm_db.get(str(_paradigm_state["fits_path"]), [])
        if not entries:
            print("No paradigm objects labeled yet for this plate.")
            return
        print(f"{len(entries)} labeled objects:")
        for e in entries:
            print(f"  {e['label']:25s} RA={e['ra']:.5f} Dec={e['dec']:.5f} ({e['source']})")

def _open_paradigm(_):
    fits_path = paradigm_plate_dropdown.value
    if fits_path is None:
        return

    data = astrofits.getdata(fits_path)
    hdr = astrofits.getheader(fits_path)
    wcs = WCS(hdr)

    _paradigm_state.update({
        "fits_path": fits_path, "wcs": wcs, "data": data,
        "xs": [], "ys": [], "manual": [],
        "scatter": None,
    })

    with paradigm_plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(data, origin="lower", cmap="gray")
        ax.set_title(f"{fits_path.name}  (left-click=label/add, right-click=remove)")
        fig.canvas.mpl_connect("button_press_event", _on_plate_click)
        _paradigm_state["fig"], _paradigm_state["ax"] = fig, ax
        _redraw_sources()
        plt.show()

    with paradigm_output:
        clear_output(wait=True)

    paradigm_status.value = f"Editing {fits_path.name}  (0 markers — click 'Auto-Load Stars' or add manually)"
    _refresh_paradigm_table()

def _auto_load_stars(_):
    if _paradigm_state["fits_path"] is None:
        with paradigm_output:
            clear_output(wait=True)
            print("No plate is currently open for labeling.")
        return

    fits_path = _paradigm_state["fits_path"]

    sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(fits_path)

    if sources is None or len(sources) == 0:
        with paradigm_output:
            clear_output(wait=True)
            print("No sources detected on this plate.")
        return

    new_xs = np.array(sources[x_col], dtype=float)
    new_ys = np.array(sources[y_col], dtype=float)

    existing_xs = np.array(_paradigm_state["xs"], dtype=float)
    existing_ys = np.array(_paradigm_state["ys"], dtype=float)

    added = 0
    skipped = 0

    for nx, ny in zip(new_xs, new_ys):
        if len(existing_xs) > 0:
            d = np.hypot(existing_xs - nx, existing_ys - ny)
            if d.min() <= PARADIGM_CLICK_PX:
                skipped += 1
                continue

        _paradigm_state["xs"].append(float(nx))
        _paradigm_state["ys"].append(float(ny))
        _paradigm_state["manual"].append(False)
        existing_xs = np.array(_paradigm_state["xs"], dtype=float)
        existing_ys = np.array(_paradigm_state["ys"], dtype=float)
        added += 1

    _redraw_sources()

    with paradigm_output:
        clear_output(wait=True)
        print(f"Auto-loaded {added} star(s) via {algorithm}. "
              f"Skipped {skipped} too close to an existing marker.")

    paradigm_status.value = (
        f"Editing {fits_path.name}  ({len(_paradigm_state['xs'])} markers)"
    )
    _refresh_paradigm_table()

def _clear_plate(_):
    if _paradigm_state["fits_path"] is None:
        with paradigm_output:
            clear_output(wait=True)
            print("No plate is currently open for labeling.")
        return

    fits_path = _paradigm_state["fits_path"]
    n_markers = len(_paradigm_state["xs"])
    n_labels = len(paradigm_db.get(str(fits_path), []))

    _paradigm_state["xs"] = []
    _paradigm_state["ys"] = []
    _paradigm_state["manual"] = []

    paradigm_db.pop(str(fits_path), None)
    save_paradigm_db()
    bump_paradigm_version()

    _redraw_sources()
    with paradigm_output:
        clear_output(wait=True)
        print(f"Cleared plate: removed {n_markers} marker(s) and {n_labels} saved label(s).")

    paradigm_status.value = f"Editing {fits_path.name}  (0 markers)"
    _refresh_paradigm_table()

def _copy_name_to_search(_):
    if _paradigm_state["fits_path"] is None:
        with paradigm_output:
            clear_output(wait=True)
            print("No plate is currently open for labeling.")
        return

    name = _paradigm_state["fits_path"].name
    search_box.value = name
    with paradigm_output:
        clear_output(wait=True)
        print(f"Copied '{name}' into the Search box above.")

def _apply_labels(_):
    """Runs the expensive full_scan() (Gaia/SIMBAD/APASS + quality
    refinement) across all plates so paradigm labels propagate everywhere
    and the main viewer below gets populated. Gated behind
    paradigm_established() -- there's nothing meaningful to apply, and no
    reason to pay for ~10k plates of network calls, until at least one
    object has actually been labeled on a paradigm plate."""
    if not paradigm_established():
        paradigm_status.value = "Label at least one object on a paradigm plate before applying."
        return
    paradigm_status.value = "Running full scan (Gaia/SIMBAD/APASS) and applying labels to all plates..."
    full_scan()
    paradigm_status.value = "Done — full scan complete, labels applied, main viewer enabled below."

auto_load_btn.on_click(_auto_load_stars)
clear_plate_btn.on_click(_clear_plate)
copy_name_btn.on_click(_copy_name_to_search)
apply_labels_btn.on_click(_apply_labels)
paradigm_plate_dropdown.observe(_open_paradigm, names="value")

paradigm_panel = widgets.VBox([
    widgets.HTML("<b>Paradigm plate labeling</b>"),
    category_filter,
    search_box,
    paradigm_plate_dropdown,
    paradigm_dropdown_status,
    widgets.HBox([auto_load_btn, clear_plate_btn, copy_name_btn, apply_labels_btn]),
    paradigm_instructions,
    paradigm_status,
    paradigm_plot_output,
    paradigm_output,
    paradigm_table_output,
])

display(paradigm_panel)
display(widgets.VBox([scan_progress_html, scan_status]))

# Run the initial pre-scan across ALL plates (fast, offline, parallelized
# across PRESCAN_WORKERS threads; uses cache -- only new/stale plates do
# real work). The expensive full_scan() only runs later, once triggered
# by 'Apply Labels' after a paradigm plate has been labeled.
prescan()

# Main viewer
progress = widgets.IntProgress(value=0, min=0, max=100, description="Progress:")
status = widgets.Label(value="Ready")
output = widgets.Output()
linearity_output = widgets.Output()

def show_plate(change):

    fits_path = dropdown.value

    if fits_path is None:
        print("No plate selected.")
        return

    paradigm_ref_indicator.value = bool(paradigm_db.get(str(fits_path)))

    meta = plate_db.get(str(fits_path))

    if meta is None or meta["render"] is None:
        with output:
            clear_output(wait=True)
            print("No data available for this plate.")
        with linearity_output:
            clear_output(wait=True)
        return

    r = meta["render"]

    if "gaia_names" not in r:
        with output:
            clear_output(wait=True)
            print("This plate hasn't been through the full scan yet. Click 'Apply Labels' above.")
        with linearity_output:
            clear_output(wait=True)
        return

    data = astrofits.getdata(fits_path)

    progress.value = 50
    status.value = "Rendering..."

    with output:
        clear_output(wait=True)

        if display_mode.value == "Object ID":
            labels = r["resolved_names"]
        else:
            labels = r["gaia_names"]

        if r["found"]:
            labels = list(labels)
            labels[r["target_idx"]] = TARGET_STAR_NAME

        sources_x = np.array(r["sources"][r["x_col"]], dtype=float)
        sources_y = np.array(r["sources"][r["y_col"]], dtype=float)
        n_sources_here = len(sources_x)

        # Defensive fallbacks for plates that haven't gone through the
        # PHOTOMETRY_VERSION-gated robust-photometry pass yet (they still
        # display fine, just without these extra diagnostics until the
        # next 'Apply Labels' click reprocesses them).
        aper_radii_arr = np.asarray(r.get("aper_radii", np.full(n_sources_here, APERTURE_RADIUS)))
        is_saturated_arr = np.asarray(r.get("is_saturated", np.zeros(n_sources_here, dtype=bool)))
        fwhm_arr = np.asarray(r.get("fwhm", np.full(n_sources_here, np.nan)))
        derived_b_arr = np.asarray(r.get("derived_b", np.full(n_sources_here, np.nan)))
        catalog_b_arr = np.asarray(r.get("catalog_b", np.full(n_sources_here, np.nan)))

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(data, origin="lower", cmap="gray")
        axes[0].set_title("Raw Plate")

        axes[1].imshow(data, origin="lower", cmap="gray")
        axes[1].set_title("Source Detections")

        axes[1].scatter(
            sources_x, sources_y,
            s=50,
            facecolors="none",
            edgecolors="red"
        )

        # Faint circle at each source's ACTUAL (possibly crowding-shrunk)
        # aperture radius, so the adaptive-aperture behavior is visible --
        # small circles in crowded fields, full-size circles for isolated
        # sources.
        for i in range(n_sources_here):
            axes[1].add_patch(plt.Circle(
                (sources_x[i], sources_y[i]), aper_radii_arr[i],
                edgecolor="red", facecolor="none", linewidth=0.5, alpha=0.35
            ))

        sat_extra_legend_handles = []
        if np.any(is_saturated_arr):
            axes[1].scatter(
                sources_x[is_saturated_arr], sources_y[is_saturated_arr],
                s=90, marker="x", color="magenta", linewidths=1.5
            )
            sat_extra_legend_handles.append(
                Line2D([0], [0], marker="x", color="magenta", linestyle="None",
                       markersize=8, label=f"Saturated ×{int(np.sum(is_saturated_arr))}")
            )

        if r["found"]:
            axes[1].scatter(
                sources_x[r["target_idx"]],
                sources_y[r["target_idx"]],
                s=120,
                edgecolors="cyan",
                facecolors="none"
            )
        elif r.get("target_fallback"):
            fb = r["target_fallback"]
            axes[1].scatter(
                [fb["x"]], [fb["y"]],
                s=160, edgecolors="cyan", facecolors="none", linewidths=2
            )
            axes[1].text(
                fb["x"] + 8, fb["y"],
                f"{TARGET_STAR_NAME} (saturated)",
                color="cyan", fontsize=6, va="center",
                bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
            )

        for i, (x, y, mag) in enumerate(zip(sources_x, sources_y, r["inst_mag"])):
            name = labels[i] if i < len(labels) else "Unknown"
            if not np.isfinite(mag):
                continue
            text = f"{name}\n{mag:.2f}"
            if aper_radii_arr[i] < DEFAULT_APERTURE_R:
                text += f"\nr={aper_radii_arr[i]:.0f}px"
            if i < len(is_saturated_arr) and is_saturated_arr[i]:
                text += "\nSAT"
                if i < len(fwhm_arr) and np.isfinite(fwhm_arr[i]):
                    text += f" FWHM={fwhm_arr[i]:.1f}"
            if (i < len(catalog_b_arr) and i < len(derived_b_arr)
                    and np.isfinite(catalog_b_arr[i]) and np.isfinite(derived_b_arr[i])):
                resid = derived_b_arr[i] - catalog_b_arr[i]
                text += f"\nΔB={resid:+.2f}"
            axes[1].text(
                x + 5, y + 5,
                text,
                color="yellow", fontsize=6,
                bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
            )

        self_limit = r.get("self_plate_limit_b")
        if self_limit is not None:
            axes[1].text(
                0.02, 0.98, f"Self plate limit (5σ): B ≈ {self_limit:.2f}",
                transform=axes[1].transAxes, color="yellow", fontsize=9,
                va="top", bbox=dict(facecolor="black", alpha=0.6, edgecolor="none")
            )

        if error_toggle.value == "Show Errors":
            annotate_errors(axes[1], r["errors"])

        # annotate_errors() (if it ran) just set its own legend as "the"
        # legend on this axes -- preserve it as a separate artist so the
        # saturated-source legend below doesn't silently replace it.
        if sat_extra_legend_handles:
            prior_legend = axes[1].get_legend()
            sat_legend = axes[1].legend(
                handles=sat_extra_legend_handles, loc="upper right", fontsize=7,
                framealpha=0.6, facecolor="black", labelcolor="white", edgecolor="gray"
            )
            if prior_legend is not None:
                axes[1].add_artist(prior_legend)

        plt.tight_layout()

        # Static PNG, not an interactive %matplotlib widget figure -- see
        # the comment in _open_review_plate() above for why: this main
        # viewer plot doesn't need clicking, and leaving it interactive
        # would (like the review panel did) silently steal click focus
        # away from the Paradigm labeling plot elsewhere in this cell.
        buf = BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        display(widgets.Image(value=buf.read(), format="png"))

        progress.value = 100
        status.value = f"Done | quality: {meta.get('quality', 'fair')}"

    with linearity_output:
        clear_output(wait=True)

        calib = r.get("calibration")
        mask = r["calib_mask"]

        x_lin = np.array(r["inst_mag"])[mask]
        y_lin = np.array(r["catalog_b"])[mask]

        fig2, ax2 = plt.subplots(figsize=(6, 5))
        ax2.scatter(y_lin, x_lin, s=20, color="steelblue", alpha=0.7, label="Matched sources")

        if calib is not None and len(y_lin) > 0:
            y_fit = np.linspace(y_lin.min(), y_lin.max(), 100)
            x_fit = calib["slope"] * y_fit + calib["intercept"]
            ax2.plot(y_fit, x_fit, color="red", linewidth=1.5,
                     label=f"Fit: slope={calib['slope']:.3f}, n={calib['n_used']}")
            residuals = x_lin - (calib["slope"] * y_lin + calib["intercept"])
            rms = np.sqrt(np.mean(residuals ** 2))
            ax2.set_title(f"Linearity Check (RMS = {rms:.3f} mag)")
        else:
            ax2.set_title("Linearity Check (not enough APASS matches)")

        ax2.set_xlabel("APASS B magnitude")
        ax2.set_ylabel("Instrumental magnitude")
        ax2.invert_yaxis()
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)

        plt.tight_layout()

        buf2 = BytesIO()
        fig2.savefig(buf2, format="png", bbox_inches="tight")
        plt.close(fig2)
        buf2.seek(0)
        display(widgets.Image(value=buf2.read(), format="png"))

        # Self-derived plate limit + differential-photometry cross-check
        # summary. Both are purely diagnostic (neither feeds back into
        # classify_plate_quality()) -- this is where to actually see them.
        self_limit = r.get("self_plate_limit_b")
        if self_limit is not None:
            print(f"Self-derived plate limit (5σ): B ≈ {self_limit:.2f}")

        diff_ref_idx = r.get("diff_ref_idx")
        derived_b_arr2 = r.get("derived_b")
        catalog_b_arr2 = np.asarray(r.get("catalog_b", []))
        if diff_ref_idx is not None and derived_b_arr2 is not None:
            derived_b_arr2 = np.asarray(derived_b_arr2)
            ref_name = labels[diff_ref_idx] if diff_ref_idx < len(labels) else "Unknown"
            print(f"Differential-photometry reference: source #{diff_ref_idx} ({ref_name})")
            both_finite = np.isfinite(derived_b_arr2) & np.isfinite(catalog_b_arr2)
            both_finite[diff_ref_idx] = False  # exclude the reference itself (residual is 0 by construction)
            if np.any(both_finite):
                resid2 = derived_b_arr2[both_finite] - catalog_b_arr2[both_finite]
                print(f"Differential-vs-APASS residuals ({int(np.sum(both_finite))} source(s)): "
                      f"mean={np.mean(resid2):+.3f}  RMS={np.sqrt(np.mean(resid2 ** 2)):.3f} mag")
        else:
            print("Differential photometry: no valid unsaturated APASS-matched reference source available.")

rebuild_dropdown()

dropdown.observe(show_plate, names="value")
display_mode.observe(show_plate, names="value")
error_toggle.observe(show_plate, names="value")

display(widgets.VBox([
    display_mode,
    error_toggle,
    paradigm_ref_indicator,
    main_category_filter,
    main_search_box,
    dropdown,
    dropdown_status,
    progress,
    status,
    output,
    linearity_output
]))

if dropdown.options:
    dropdown.value = dropdown.options[0][1]

Plate-limit reference: median lim_mag_apass=13.479, median lim_mag_atlas=13.408000000000001 (from 2711 / 2700 plates with data)


Pre-scan autosaved 500 plates
Pre-scan autosaved 1000 plates
Pre-scan autosaved 1500 plates
Pre-scan autosaved 2000 plates
Pre-scan autosaved 2500 plates
Pre-scan autosaved 3000 plates
